# FiQA BM25 + Compressed Dense RRF Benchmark

## Goal
Evaluate whether BM25 can recover relevant documents missed by compressed dense ANN retrieval.

### Compared methods
1. BM25 only
2. Float32 exact dense retrieval
3. IVF-PQ ADC, M=96, nprobe=16
4. Native Faiss OPQMatrix + IVF-PQ ADC, M=96, nprobe=16
5. BM25 + IVF-PQ RRF
6. BM25 + OPQ-IVF-PQ RRF

### Evaluation discipline
- FiQA queries are deterministically split into calibration (20%) and held-out (80%).
- This first run uses a fixed RRF configuration: `rrf_k=60`, equal weights.
- The held-out split is the primary result for later claims.
- No RRF tuning is performed in this notebook.


In [ ]:
# 1. Install dependencies
%pip uninstall -y faiss-cpu faiss-gpu faiss-gpu-cu11 faiss-gpu-cu12

%pip install -q --upgrade --upgrade-strategy only-if-needed \
  "transformers==4.49.0" \
  "sentence-transformers==3.4.1" \
  "tokenizers>=0.21,<0.22" \
  "sentencepiece>=0.2.0" \
  "safetensors>=0.4.5" \
  "faiss-gpu-cu12" \
  "rank-bm25==0.2.2" \
  "pandas==2.2.2" \
  "matplotlib>=3.7"

print("Install complete.")

In [ ]:
# 2. Imports, configuration, and reproducibility
import csv
import gc
import hashlib
import json
import random
import re
import time
import urllib.request
import zipfile
from pathlib import Path

import faiss
import faiss.contrib.torch_utils
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. In Colab, select a T4 GPU runtime.")

DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if faiss.get_num_gpus() < 1:
    raise RuntimeError(
        "Faiss GPU backend was not detected. Restart the runtime, then rerun from Cell 1."
    )

FIQA_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
DATA_DIR = Path("beir_data")
FIQA_DIR = DATA_DIR / "fiqa"
RESULT_DIR = Path("results/hybrid_bm25_rrf_fiqa")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
DOC_EMBED_BATCH_SIZE = 256
QUERY_EMBED_BATCH_SIZE = 256

CANDIDATE_K = 100
FINAL_K = 10

FAISS_NLIST = 256
FAISS_NPROBE = 16
PQ_M = 96
PQ_NBITS = 8
FAISS_TRAIN_POINTS = 24_000
FAISS_GPU_BATCH_SIZE = 64

RRF_K = 60
RRF_W_SPARSE = 1.0
RRF_W_DENSE = 1.0

CALIBRATION_MODULUS = 5   # deterministic 20% calibration split
CALIBRATION_BUCKET = 0

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Faiss:", faiss.__version__)
print("Faiss visible GPUs:", faiss.get_num_gpus())

In [ ]:
# 3. Load FiQA and create deterministic calibration / held-out split

def download_and_extract_fiqa():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / "fiqa.zip"

    if not FIQA_DIR.exists():
        if not zip_path.exists():
            print("Downloading FiQA...")
            urllib.request.urlretrieve(FIQA_URL, zip_path)

        print("Extracting FiQA...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DATA_DIR)

    required = [
        FIQA_DIR / "corpus.jsonl",
        FIQA_DIR / "queries.jsonl",
        FIQA_DIR / "qrels" / "test.tsv",
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"FiQA extraction incomplete: {missing}")


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def read_qrels(path):
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            qid = str(row["query-id"])
            docid = str(row["corpus-id"])
            score = int(row["score"])
            result.setdefault(qid, {})[docid] = score
    return result


def split_name(qid):
    bucket = int(hashlib.sha256(qid.encode("utf-8")).hexdigest(), 16) % CALIBRATION_MODULUS
    return "calibration" if bucket == CALIBRATION_BUCKET else "heldout"


download_and_extract_fiqa()

corpus_rows = read_jsonl(FIQA_DIR / "corpus.jsonl")
query_rows = read_jsonl(FIQA_DIR / "queries.jsonl")
all_qrels = read_qrels(FIQA_DIR / "qrels" / "test.tsv")

doc_ids = [str(row["_id"]) for row in corpus_rows]
doc_texts = [
    ((row.get("title") or "") + "\n" + (row.get("text") or "")).strip()
    for row in corpus_rows
]
doc_id_to_index = {doc_id: i for i, doc_id in enumerate(doc_ids)}

query_ids, query_texts, qrels = [], [], {}
for row in query_rows:
    qid = str(row["_id"])
    if qid not in all_qrels:
        continue

    relevant = {
        doc_id: score
        for doc_id, score in all_qrels[qid].items()
        if doc_id in doc_id_to_index and score > 0
    }
    if relevant:
        query_ids.append(qid)
        query_texts.append(str(row["text"]))
        qrels[qid] = relevant

query_splits = np.array([split_name(qid) for qid in query_ids])
calibration_mask = query_splits == "calibration"
heldout_mask = query_splits == "heldout"

print(f"Documents: {len(doc_ids):,}")
print(f"Evaluation queries: {len(query_ids):,}")
print(f"Calibration queries: {int(calibration_mask.sum()):,}")
print(f"Held-out queries: {int(heldout_mask.sum()):,}")

In [ ]:
# 4. BM25, embedding generation, Faiss retrieval, RRF, and evaluation helpers

TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+")

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


def encode_texts_gpu(texts, model_name, batch_size):
    model = SentenceTransformer(model_name, device="cuda")
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cuda",
    )
    del model
    torch.cuda.empty_cache()
    return torch.from_numpy(vectors.astype(np.float32)).to(DEVICE)


def per_query_metrics(rankings, query_ids, doc_ids, qrels, k=10):
    recall = np.zeros(len(query_ids), dtype=np.float64)
    mrr = np.zeros(len(query_ids), dtype=np.float64)
    ndcg = np.zeros(len(query_ids), dtype=np.float64)

    discounts = 1.0 / np.log2(np.arange(2, k + 2))

    for i, (row, qid) in enumerate(zip(rankings[:, :k], query_ids)):
        relevance_map = qrels[qid]
        relevant = set(relevance_map)

        retrieved_ids = [
            doc_ids[int(idx)] if int(idx) >= 0 else None
            for idx in row
        ]
        retrieved_set = {docid for docid in retrieved_ids if docid is not None}
        recall[i] = len(relevant & retrieved_set) / len(relevant)

        gains = np.array(
            [relevance_map.get(docid, 0) if docid is not None else 0 for docid in retrieved_ids],
            dtype=np.float64,
        )

        first_rel = np.where(gains > 0)[0]
        mrr[i] = 1.0 / (first_rel[0] + 1) if len(first_rel) else 0.0

        dcg = np.sum((2**gains - 1) * discounts)
        ideal = np.sort(np.asarray(list(relevance_map.values()), dtype=np.float64))[::-1][:k]
        idcg = np.sum((2**ideal - 1) * discounts[:len(ideal)])
        ndcg[i] = dcg / idcg if idcg > 0 else 0.0

    return {
        "recall_at_10": recall,
        "mrr_at_10": mrr,
        "ndcg_at_10": ndcg,
    }


def aggregate_metrics(per_query, mask):
    return {
        name: float(values[mask].mean())
        for name, values in per_query.items()
    }


def bm25_search(bm25, texts, k):
    rankings = []
    start = time.perf_counter()

    for text in texts:
        scores = bm25.get_scores(tokenize(text))
        candidate = np.argpartition(scores, -k)[-k:]
        candidate = candidate[np.argsort(scores[candidate])[::-1]]
        rankings.append(candidate.astype(np.int64))

    elapsed = time.perf_counter() - start
    return np.vstack(rankings), elapsed


def gpu_search(index, queries_gpu, k, batch_size=FAISS_GPU_BATCH_SIZE):
    for _ in range(3):
        index.search(queries_gpu[:min(batch_size, len(queries_gpu))].contiguous(), k)

    torch.cuda.synchronize()
    rankings = []
    batch_per_query_ms = []
    timed_queries = 0
    timed_seconds = 0.0

    for start in range(0, len(queries_gpu), batch_size):
        batch = queries_gpu[start:start + batch_size].contiguous()

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _, ids = index.search(batch, k)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        if isinstance(ids, torch.Tensor):
            ids = ids.detach().cpu().numpy()
        else:
            ids = np.asarray(ids)

        rankings.append(ids)

        if len(batch) == batch_size:
            timed_seconds += elapsed
            timed_queries += len(batch)
            batch_per_query_ms.append(elapsed * 1000.0 / len(batch))

    return (
        np.concatenate(rankings, axis=0),
        {
            "search_seconds": float(timed_seconds),
            "timed_query_count": int(timed_queries),
            "p50_latency_ms": float(np.median(batch_per_query_ms)),
            "p95_latency_ms": float(np.percentile(batch_per_query_ms, 95)),
            "queries_per_second": float(timed_queries / timed_seconds),
        },
    )


def rrf_fuse(sparse_rankings, dense_rankings, rrf_k=RRF_K,
             w_sparse=RRF_W_SPARSE, w_dense=RRF_W_DENSE, final_k=FINAL_K):
    fused = np.empty((len(sparse_rankings), final_k), dtype=np.int64)

    for i, (sparse_row, dense_row) in enumerate(zip(sparse_rankings, dense_rankings)):
        scores = {}

        for rank, doc_idx in enumerate(sparse_row, start=1):
            doc_idx = int(doc_idx)
            scores[doc_idx] = scores.get(doc_idx, 0.0) + w_sparse / (rrf_k + rank)

        for rank, doc_idx in enumerate(dense_row, start=1):
            doc_idx = int(doc_idx)
            scores[doc_idx] = scores.get(doc_idx, 0.0) + w_dense / (rrf_k + rank)

        ordered = sorted(scores.items(), key=lambda item: (-item[1], item[0]))
        fused[i] = [doc_idx for doc_idx, _ in ordered[:final_k]]

    return fused


def mean_candidate_overlap(left, right):
    return float(np.mean([
        len(set(a.tolist()) & set(b.tolist())) / CANDIDATE_K
        for a, b in zip(left, right)
    ]))


def win_loss_tie(reference_ndcg, hybrid_ndcg, eps=1e-12):
    delta = hybrid_ndcg - reference_ndcg
    return {
        "wins": int(np.sum(delta > eps)),
        "losses": int(np.sum(delta < -eps)),
        "ties": int(np.sum(np.abs(delta) <= eps)),
        "mean_delta_ndcg_at_10": float(np.mean(delta)),
    }

In [ ]:
# 5. Build indexes and run the complete benchmark

print("Building BM25 index...")
bm25_build_start = time.perf_counter()
bm25 = BM25Okapi([tokenize(text) for text in doc_texts])
bm25_build_seconds = time.perf_counter() - bm25_build_start

print("Encoding documents...")
X_docs = encode_texts_gpu(doc_texts, EMBEDDING_MODEL, DOC_EMBED_BATCH_SIZE)

print("Encoding queries...")
X_queries = encode_texts_gpu(query_texts, EMBEDDING_MODEL, QUERY_EMBED_BATCH_SIZE)

N_DOCS, D = X_docs.shape
assert D % PQ_M == 0, f"Embedding dimension {D} must be divisible by M={PQ_M}"

X_docs_gpu = X_docs.contiguous().float()
X_queries_gpu = X_queries.contiguous().float()

train_count = min(FAISS_TRAIN_POINTS, N_DOCS)
train_indices = torch.randperm(N_DOCS, device=DEVICE)[:train_count]
X_train_gpu = X_docs_gpu[train_indices].contiguous()

gpu_resources = faiss.StandardGpuResources()
gpu_resources.setDefaultNullStreamAllDevices()
gpu_resources.setTempMemory(768 * 1024 * 1024)

flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
flat_config.useFloat16 = False

ivfpq_config = faiss.GpuIndexIVFPQConfig()
ivfpq_config.device = 0
ivfpq_config.useFloat16LookupTables = True
ivfpq_config.indicesOptions = faiss.INDICES_32_BIT
ivfpq_config.interleavedLayout = True

print("Running BM25...")
bm25_rankings, bm25_latency = bm25_search(bm25, query_texts, CANDIDATE_K)

print("Building exact Float32 GPU baseline...")
gpu_flat = faiss.GpuIndexFlatIP(gpu_resources, D, flat_config)
gpu_flat.add(X_docs_gpu)
flat_rankings, flat_latency = gpu_search(gpu_flat, X_queries_gpu, CANDIDATE_K)

print("Training IVF-PQ M=96...")
pq_index = faiss.GpuIndexIVFPQ(
    gpu_resources, D, FAISS_NLIST, PQ_M, PQ_NBITS,
    faiss.METRIC_INNER_PRODUCT, ivfpq_config
)
pq_index.train(X_train_gpu)
pq_index.add(X_docs_gpu)
pq_index.nprobe = FAISS_NPROBE
pq_rankings, pq_latency = gpu_search(pq_index, X_queries_gpu, CANDIDATE_K)

print("Training native Faiss OPQMatrix + IVF-PQ M=96...")
train_np = np.ascontiguousarray(X_train_gpu.detach().cpu().numpy().astype(np.float32))
docs_np = np.ascontiguousarray(X_docs_gpu.detach().cpu().numpy().astype(np.float32))
queries_np = np.ascontiguousarray(X_queries_gpu.detach().cpu().numpy().astype(np.float32))

native_opq = faiss.OPQMatrix(D, PQ_M)
native_opq.niter = 25
native_opq.niter_pq = 4
native_opq.train(train_np)

docs_rot_np = np.ascontiguousarray(native_opq.apply_py(docs_np).astype(np.float32))
queries_rot_np = np.ascontiguousarray(native_opq.apply_py(queries_np).astype(np.float32))
faiss.normalize_L2(docs_rot_np)
faiss.normalize_L2(queries_rot_np)

docs_rot_gpu = torch.from_numpy(docs_rot_np).to(DEVICE).contiguous()
queries_rot_gpu = torch.from_numpy(queries_rot_np).to(DEVICE).contiguous()
opq_train_gpu = docs_rot_gpu[train_indices].contiguous()

opq_index = faiss.GpuIndexIVFPQ(
    gpu_resources, D, FAISS_NLIST, PQ_M, PQ_NBITS,
    faiss.METRIC_INNER_PRODUCT, ivfpq_config
)
opq_index.train(opq_train_gpu)
opq_index.add(docs_rot_gpu)
opq_index.nprobe = FAISS_NPROBE
opq_rankings, opq_latency = gpu_search(opq_index, queries_rot_gpu, CANDIDATE_K)

fusion_start = time.perf_counter()
bm25_pq_rrf_rankings = rrf_fuse(bm25_rankings, pq_rankings)
pq_rrf_fusion_seconds = time.perf_counter() - fusion_start

fusion_start = time.perf_counter()
bm25_opq_rrf_rankings = rrf_fuse(bm25_rankings, opq_rankings)
opq_rrf_fusion_seconds = time.perf_counter() - fusion_start

methods = {
    "bm25": {"rankings": bm25_rankings, "latency": bm25_latency},
    "float32_flat_ip": {"rankings": flat_rankings, "latency": flat_latency},
    "ivfpq_m96_np16": {"rankings": pq_rankings, "latency": pq_latency},
    "opq_ivfpq_m96_np16": {"rankings": opq_rankings, "latency": opq_latency},
    "bm25_ivfpq_rrf": {
        "rankings": bm25_pq_rrf_rankings,
        "latency": {
            "search_seconds": float(
                bm25_latency + pq_latency["search_seconds"] + pq_rrf_fusion_seconds
            ),
            "component_latency": "sequential_bm25_plus_ivfpq_plus_rrf",
        },
    },
    "bm25_opq_ivfpq_rrf": {
        "rankings": bm25_opq_rrf_rankings,
        "latency": {
            "search_seconds": float(
                bm25_latency + opq_latency["search_seconds"] + opq_rrf_fusion_seconds
            ),
            "component_latency": "sequential_bm25_plus_opq_ivfpq_plus_rrf",
        },
    },
}

summary_rows = []
per_query_frames = []

for method, payload in methods.items():
    rankings = payload["rankings"]
    scores = per_query_metrics(rankings, query_ids, doc_ids, qrels, FINAL_K)

    for split, mask in {
        "all": np.ones(len(query_ids), dtype=bool),
        "calibration": calibration_mask,
        "heldout": heldout_mask,
    }.items():
        row = {
            "method": method,
            "split": split,
            "query_count": int(mask.sum()),
            **aggregate_metrics(scores, mask),
        }

        if method == "bm25":
            row.update({
                "candidate_k": CANDIDATE_K,
                "bm25_build_seconds": float(bm25_build_seconds),
                "sparse_search_seconds_all_queries": float(bm25_latency),
                "sparse_latency_per_query_ms_all_queries": float(bm25_latency * 1000 / len(query_ids)),
            })
        elif method in {"float32_flat_ip", "ivfpq_m96_np16", "opq_ivfpq_m96_np16"}:
            row.update(payload["latency"])
        else:
            row.update(payload["latency"])

        summary_rows.append(row)

    frame = pd.DataFrame({
        "query_id": query_ids,
        "split": query_splits,
        "method": method,
        "recall_at_10": scores["recall_at_10"],
        "mrr_at_10": scores["mrr_at_10"],
        "ndcg_at_10": scores["ndcg_at_10"],
    })
    per_query_frames.append(frame)

summary_df = pd.DataFrame(summary_rows)
per_query_df = pd.concat(per_query_frames, ignore_index=True)

analysis_rows = []
for dense_method, hybrid_method, dense_rankings, hybrid_rankings in [
    ("ivfpq_m96_np16", "bm25_ivfpq_rrf", pq_rankings, bm25_pq_rrf_rankings),
    ("opq_ivfpq_m96_np16", "bm25_opq_ivfpq_rrf", opq_rankings, bm25_opq_rrf_rankings),
]:
    # Each method is appended in the same original query order.
    # Keep that order so calibration_mask / heldout_mask remain aligned.
    dense_ndcg = per_query_df[
        per_query_df["method"] == dense_method
    ]["ndcg_at_10"].to_numpy()

    hybrid_ndcg = per_query_df[
        per_query_df["method"] == hybrid_method
    ]["ndcg_at_10"].to_numpy()

    for split, mask in {
        "all": np.ones(len(query_ids), dtype=bool),
        "calibration": calibration_mask,
        "heldout": heldout_mask,
    }.items():
        stats = win_loss_tie(dense_ndcg[mask], hybrid_ndcg[mask])
        analysis_rows.append({
            "dense_method": dense_method,
            "hybrid_method": hybrid_method,
            "split": split,
            "query_count": int(mask.sum()),
            "bm25_dense_candidate_overlap_at_100": mean_candidate_overlap(
                bm25_rankings[mask],
                dense_rankings[mask],
            ),
            **stats,
        })

analysis_df = pd.DataFrame(analysis_rows)

summary_df.to_csv(RESULT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
per_query_df.to_csv(RESULT_DIR / "per_query_metrics.csv", index=False, encoding="utf-8-sig")
analysis_df.to_csv(RESULT_DIR / "hybrid_analysis.csv", index=False, encoding="utf-8-sig")

metadata = {
    "dataset": "FiQA-2018 / BEIR",
    "embedding_model": EMBEDDING_MODEL,
    "corpus_documents": int(N_DOCS),
    "evaluation_queries": int(len(query_ids)),
    "candidate_k": CANDIDATE_K,
    "final_k": FINAL_K,
    "rrf": {
        "rrf_k": RRF_K,
        "w_sparse": RRF_W_SPARSE,
        "w_dense": RRF_W_DENSE,
        "tuned": False,
    },
    "split": {
        "calibration_rule": "sha256(query_id) mod 5 == 0",
        "calibration_queries": int(calibration_mask.sum()),
        "heldout_queries": int(heldout_mask.sum()),
    },
    "faiss": {
        "nlist": FAISS_NLIST,
        "nprobe": FAISS_NPROBE,
        "m": PQ_M,
        "nbits": PQ_NBITS,
        "native_opq": True,
    },
    "timing_note": (
        "BM25 is measured as sequential CPU scoring over the full corpus. "
        "Dense timing is synchronized GPU search timing. Hybrid sequential time "
        "combines sparse retrieval, dense retrieval, and CPU-side RRF fusion; "
        "it is not an end-to-end API latency claim."
    ),
}
(RESULT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

heldout_summary = (
    summary_df[summary_df["split"] == "heldout"]
    [["method", "query_count", "recall_at_10", "mrr_at_10", "ndcg_at_10"]]
    .sort_values("ndcg_at_10", ascending=False)
)

print("\nHeld-out quality results")
display(heldout_summary)

print("\nHybrid overlap and win/loss/tie analysis")
display(analysis_df[analysis_df["split"] == "heldout"])

print("\nSaved outputs:")
for path in sorted(RESULT_DIR.iterdir()):
    print("-", path)

del train_np, docs_np, queries_np, docs_rot_np, queries_rot_np
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 6. Calibration-only RRF tuning, followed by one held-out evaluation
#
# Selection uses calibration queries only.
# Held-out queries are evaluated only after each dense backend's configuration
# has been frozen.

RRF_K_GRID = [20, 60, 100]
DENSE_WEIGHT_GRID = [1.0, 1.5, 2.0, 3.0, 4.0]
SPARSE_WEIGHT = 1.0

def evaluate_rrf_grid(dense_name, dense_rankings):
    calibration_rows = []

    for rrf_k in RRF_K_GRID:
        for w_dense in DENSE_WEIGHT_GRID:
            fused = rrf_fuse(
                bm25_rankings,
                dense_rankings,
                rrf_k=rrf_k,
                w_sparse=SPARSE_WEIGHT,
                w_dense=w_dense,
                final_k=FINAL_K,
            )

            scores = per_query_metrics(
                fused,
                query_ids,
                doc_ids,
                qrels,
                FINAL_K,
            )

            calibration_metrics = aggregate_metrics(
                scores,
                calibration_mask,
            )

            calibration_rows.append({
                "dense_backend": dense_name,
                "rrf_k": int(rrf_k),
                "w_sparse": float(SPARSE_WEIGHT),
                "w_dense": float(w_dense),
                "calibration_query_count": int(calibration_mask.sum()),
                **calibration_metrics,
            })

    calibration_df = pd.DataFrame(calibration_rows)

    # Deterministic selection:
    # primary nDCG@10, then MRR@10, Recall@10, then lower dense weight / rrf_k.
    selected = (
        calibration_df
        .sort_values(
            [
                "ndcg_at_10",
                "mrr_at_10",
                "recall_at_10",
                "w_dense",
                "rrf_k",
            ],
            ascending=[False, False, False, True, True],
        )
        .iloc[0]
        .to_dict()
    )

    selected_rankings = rrf_fuse(
        bm25_rankings,
        dense_rankings,
        rrf_k=int(selected["rrf_k"]),
        w_sparse=float(selected["w_sparse"]),
        w_dense=float(selected["w_dense"]),
        final_k=FINAL_K,
    )

    selected_scores = per_query_metrics(
        selected_rankings,
        query_ids,
        doc_ids,
        qrels,
        FINAL_K,
    )

    heldout_metrics = aggregate_metrics(
        selected_scores,
        heldout_mask,
    )

    heldout_row = {
        "dense_backend": dense_name,
        "selected_on": "calibration_only",
        "heldout_query_count": int(heldout_mask.sum()),
        "rrf_k": int(selected["rrf_k"]),
        "w_sparse": float(selected["w_sparse"]),
        "w_dense": float(selected["w_dense"]),
        **heldout_metrics,
    }

    return calibration_df, heldout_row, selected_rankings, selected_scores


pq_grid_df, pq_heldout_row, pq_tuned_rankings, pq_tuned_scores = (
    evaluate_rrf_grid("ivfpq_m96_np16", pq_rankings)
)

opq_grid_df, opq_heldout_row, opq_tuned_rankings, opq_tuned_scores = (
    evaluate_rrf_grid("opq_ivfpq_m96_np16", opq_rankings)
)

tuning_grid_df = pd.concat(
    [pq_grid_df, opq_grid_df],
    ignore_index=True,
)

tuned_heldout_df = pd.DataFrame(
    [pq_heldout_row, opq_heldout_row]
)

dense_heldout_df = (
    summary_df[
        (summary_df["split"] == "heldout")
        & summary_df["method"].isin(
            ["ivfpq_m96_np16", "opq_ivfpq_m96_np16"]
        )
    ][
        [
            "method",
            "recall_at_10",
            "mrr_at_10",
            "ndcg_at_10",
        ]
    ]
    .rename(columns={"method": "dense_backend"})
)

comparison_df = tuned_heldout_df.merge(
    dense_heldout_df,
    on="dense_backend",
    suffixes=("_tuned_rrf", "_dense_only"),
)

for metric in ["recall_at_10", "mrr_at_10", "ndcg_at_10"]:
    comparison_df[f"delta_{metric}"] = (
        comparison_df[f"{metric}_tuned_rrf"]
        - comparison_df[f"{metric}_dense_only"]
    )

selected_per_query_df = pd.concat(
    [
        pd.DataFrame({
            "query_id": query_ids,
            "split": query_splits,
            "method": "bm25_ivfpq_rrf_tuned",
            "recall_at_10": pq_tuned_scores["recall_at_10"],
            "mrr_at_10": pq_tuned_scores["mrr_at_10"],
            "ndcg_at_10": pq_tuned_scores["ndcg_at_10"],
        }),
        pd.DataFrame({
            "query_id": query_ids,
            "split": query_splits,
            "method": "bm25_opq_ivfpq_rrf_tuned",
            "recall_at_10": opq_tuned_scores["recall_at_10"],
            "mrr_at_10": opq_tuned_scores["mrr_at_10"],
            "ndcg_at_10": opq_tuned_scores["ndcg_at_10"],
        }),
    ],
    ignore_index=True,
)

tuning_grid_df.to_csv(
    RESULT_DIR / "rrf_tuning_calibration_grid.csv",
    index=False,
    encoding="utf-8-sig",
)
tuned_heldout_df.to_csv(
    RESULT_DIR / "rrf_tuning_selected_heldout.csv",
    index=False,
    encoding="utf-8-sig",
)
comparison_df.to_csv(
    RESULT_DIR / "rrf_tuning_dense_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)
selected_per_query_df.to_csv(
    RESULT_DIR / "rrf_tuning_selected_per_query.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Calibration leaderboard: IVF-PQ")
display(
    pq_grid_df.sort_values(
        ["ndcg_at_10", "mrr_at_10", "recall_at_10"],
        ascending=False,
    ).head(10)
)

print("Calibration leaderboard: OPQ-IVF-PQ")
display(
    opq_grid_df.sort_values(
        ["ndcg_at_10", "mrr_at_10", "recall_at_10"],
        ascending=False,
    ).head(10)
)

print("Frozen selected configurations, evaluated once on held-out FiQA")
display(
    comparison_df[
        [
            "dense_backend",
            "rrf_k",
            "w_sparse",
            "w_dense",
            "recall_at_10_dense_only",
            "recall_at_10_tuned_rrf",
            "delta_recall_at_10",
            "mrr_at_10_dense_only",
            "mrr_at_10_tuned_rrf",
            "delta_mrr_at_10",
            "ndcg_at_10_dense_only",
            "ndcg_at_10_tuned_rrf",
            "delta_ndcg_at_10",
        ]
    ]
)


In [ ]:
# 7. Paired bootstrap confidence intervals on held-out FiQA
#
# Uses only the frozen configurations selected in Cell 6.
# The held-out subset is never used to select RRF hyperparameters.

BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 42

def paired_bootstrap_ci(baseline, candidate, n_bootstrap, seed):
    baseline = np.asarray(baseline, dtype=np.float64)
    candidate = np.asarray(candidate, dtype=np.float64)

    if baseline.shape != candidate.shape or baseline.ndim != 1:
        raise ValueError("Baseline and candidate must be aligned 1D arrays.")

    rng = np.random.default_rng(seed)
    n = len(baseline)

    sample_indices = rng.integers(
        0,
        n,
        size=(n_bootstrap, n),
        dtype=np.int32,
    )

    deltas = (
        candidate[sample_indices].mean(axis=1)
        - baseline[sample_indices].mean(axis=1)
    )

    return {
        "point_delta": float(candidate.mean() - baseline.mean()),
        "ci95_low": float(np.percentile(deltas, 2.5)),
        "ci95_high": float(np.percentile(deltas, 97.5)),
        "p_bootstrap_improvement": float(np.mean(deltas > 0.0)),
        "p_bootstrap_two_sided_tail": float(
            min(1.0, 2.0 * min(
                np.mean(deltas <= 0.0),
                np.mean(deltas >= 0.0),
            ))
        ),
    }


comparisons = [
    ("ivfpq_m96_np16", "bm25_ivfpq_rrf_tuned"),
    ("opq_ivfpq_m96_np16", "bm25_opq_ivfpq_rrf_tuned"),
]

bootstrap_rows = []

for comparison_id, (dense_method, tuned_method) in enumerate(comparisons):
    dense_df = per_query_df[
        per_query_df["method"] == dense_method
    ][
        ["query_id", "split", "recall_at_10", "mrr_at_10", "ndcg_at_10"]
    ].copy()

    tuned_df = selected_per_query_df[
        selected_per_query_df["method"] == tuned_method
    ][
        ["query_id", "split", "recall_at_10", "mrr_at_10", "ndcg_at_10"]
    ].copy()

    joined = dense_df.merge(
        tuned_df,
        on=["query_id", "split"],
        suffixes=("_dense_only", "_tuned_rrf"),
        validate="one_to_one",
    )

    heldout = joined[joined["split"] == "heldout"].copy()

    for metric_id, metric in enumerate(
        ["recall_at_10", "mrr_at_10", "ndcg_at_10"]
    ):
        result = paired_bootstrap_ci(
            baseline=heldout[f"{metric}_dense_only"].to_numpy(),
            candidate=heldout[f"{metric}_tuned_rrf"].to_numpy(),
            n_bootstrap=BOOTSTRAP_SAMPLES,
            seed=BOOTSTRAP_SEED + comparison_id * 10 + metric_id,
        )

        bootstrap_rows.append({
            "dataset": "FiQA-2018 / BEIR",
            "evaluation_split": "heldout",
            "dense_backend": dense_method,
            "candidate_method": tuned_method,
            "metric": metric,
            "query_count": int(len(heldout)),
            "bootstrap_samples": BOOTSTRAP_SAMPLES,
            **result,
        })

fiqa_bootstrap_df = pd.DataFrame(bootstrap_rows)
fiqa_bootstrap_df.to_csv(
    RESULT_DIR / "rrf_tuning_paired_bootstrap_ci.csv",
    index=False,
    encoding="utf-8-sig",
)

print("FiQA held-out paired bootstrap confidence intervals")
display(fiqa_bootstrap_df)


In [ ]:
# 8. Export Top-100 rankings and calibration-only rank-flip risk labels
#
# Requires Cell 5 to have completed successfully.
# This cell does not retrain embeddings or indexes.

RANK_FLIP_EXPORT_K = 100
TOPK_CUTOFF = 10
BOUNDARY_WINDOW_START = 8
BOUNDARY_WINDOW_END = 12

RANK_FLIP_DIR = RESULT_DIR / "rank_flip_analysis"
RANK_FLIP_DIR.mkdir(parents=True, exist_ok=True)


def faiss_search_with_scores(index, queries_gpu, k, batch_size=FAISS_GPU_BATCH_SIZE):
    """Return Faiss scores and IDs for every query without timing claims."""
    all_scores = []
    all_ids = []

    for start in range(0, len(queries_gpu), batch_size):
        batch = queries_gpu[start:start + batch_size].contiguous()
        scores, ids = index.search(batch, k)

        if isinstance(scores, torch.Tensor):
            scores = scores.detach().cpu().numpy()
        else:
            scores = np.asarray(scores)

        if isinstance(ids, torch.Tensor):
            ids = ids.detach().cpu().numpy()
        else:
            ids = np.asarray(ids)

        all_scores.append(scores.astype(np.float32, copy=False))
        all_ids.append(ids.astype(np.int64, copy=False))

    return np.concatenate(all_scores, axis=0), np.concatenate(all_ids, axis=0)


def ranking_rows(backend, scores, rankings):
    rows = []

    for query_pos, (qid, split, score_row, rank_row) in enumerate(
        zip(query_ids, query_splits, scores, rankings)
    ):
        relevance_map = qrels[qid]

        for rank, (score, doc_idx) in enumerate(
            zip(score_row, rank_row),
            start=1,
        ):
            doc_idx = int(doc_idx)

            if doc_idx < 0:
                continue

            doc_id = doc_ids[doc_idx]
            qrel_score = int(relevance_map.get(doc_id, 0))

            rows.append({
                "query_position": int(query_pos),
                "query_id": qid,
                "split": split,
                "backend": backend,
                "rank": int(rank),
                "document_index": doc_idx,
                "document_id": doc_id,
                "retrieval_score": float(score),
                "qrel_score": qrel_score,
                "is_relevant": bool(qrel_score > 0),
            })

    return rows


print("Retrieving exact Float32 Top-100...")
exact_scores, exact_rankings = faiss_search_with_scores(
    gpu_flat,
    X_queries_gpu,
    RANK_FLIP_EXPORT_K,
)

print("Retrieving IVF-PQ Top-100...")
pq_index.nprobe = FAISS_NPROBE
pq_scores, pq_rankings_top100 = faiss_search_with_scores(
    pq_index,
    X_queries_gpu,
    RANK_FLIP_EXPORT_K,
)

print("Retrieving OPQ-IVF-PQ Top-100...")
opq_index.nprobe = FAISS_NPROBE
opq_scores, opq_rankings_top100 = faiss_search_with_scores(
    opq_index,
    queries_rot_gpu,
    RANK_FLIP_EXPORT_K,
)

rankings_by_backend = {
    "float32_flat_ip_exact": (exact_scores, exact_rankings),
    "ivfpq_m96_np16": (pq_scores, pq_rankings_top100),
    "opq_ivfpq_m96_np16": (opq_scores, opq_rankings_top100),
}

ranking_export_rows = []
for backend, (scores, rankings) in rankings_by_backend.items():
    ranking_export_rows.extend(
        ranking_rows(backend, scores, rankings)
    )

rankings_long_df = pd.DataFrame(ranking_export_rows)
rankings_long_df.to_csv(
    RANK_FLIP_DIR / "rankings_long.csv",
    index=False,
    encoding="utf-8-sig",
)

event_rows = []
boundary_rows = []

for query_pos, (qid, split) in enumerate(zip(query_ids, query_splits)):
    relevance_map = qrels[qid]

    exact_ids = exact_rankings[query_pos]
    exact_scores_row = exact_scores[query_pos]
    exact_top10 = [int(x) for x in exact_ids[:TOPK_CUTOFF] if int(x) >= 0]
    exact_top10_set = set(exact_top10)

    exact_rank_map = {
        int(doc_idx): rank
        for rank, doc_idx in enumerate(exact_ids, start=1)
        if int(doc_idx) >= 0
    }

    exact_score_map = {
        int(doc_idx): float(score)
        for doc_idx, score in zip(exact_ids, exact_scores_row)
        if int(doc_idx) >= 0
    }

    exact_margin = float(
        exact_scores_row[TOPK_CUTOFF - 1] - exact_scores_row[TOPK_CUTOFF]
    )

    for backend, scores, rankings in [
        ("ivfpq_m96_np16", pq_scores, pq_rankings_top100),
        ("opq_ivfpq_m96_np16", opq_scores, opq_rankings_top100),
    ]:
        compressed_ids = rankings[query_pos]
        compressed_scores_row = scores[query_pos]
        compressed_top10 = [
            int(x)
            for x in compressed_ids[:TOPK_CUTOFF]
            if int(x) >= 0
        ]
        compressed_top10_set = set(compressed_top10)

        compressed_rank_map = {
            int(doc_idx): rank
            for rank, doc_idx in enumerate(compressed_ids, start=1)
            if int(doc_idx) >= 0
        }

        compressed_score_map = {
            int(doc_idx): float(score)
            for doc_idx, score in zip(compressed_ids, compressed_scores_row)
            if int(doc_idx) >= 0
        }

        compressed_margin = float(
            compressed_scores_row[TOPK_CUTOFF - 1]
            - compressed_scores_row[TOPK_CUTOFF]
        )

        exact_relevant = {
            doc_idx
            for doc_idx in exact_top10_set
            if relevance_map.get(doc_ids[doc_idx], 0) > 0
        }

        compressed_relevant = {
            doc_idx
            for doc_idx in compressed_top10_set
            if relevance_map.get(doc_ids[doc_idx], 0) > 0
        }

        relevant_drops = exact_relevant - compressed_top10_set
        relevant_recoveries = compressed_relevant - exact_top10_set
        shared_relevant = exact_relevant & compressed_relevant

        nonrelevant_intrusions = {
            doc_idx
            for doc_idx in (compressed_top10_set - exact_top10_set)
            if relevance_map.get(doc_ids[doc_idx], 0) <= 0
        }

        for event_type, doc_indices in [
            ("relevant_drop", relevant_drops),
            ("nonrelevant_intrusion", nonrelevant_intrusions),
            ("relevant_recovery", relevant_recoveries),
            ("shared_relevant", shared_relevant),
        ]:
            for doc_idx in sorted(doc_indices):
                event_rows.append({
                    "query_position": int(query_pos),
                    "query_id": qid,
                    "split": split,
                    "backend": backend,
                    "event_type": event_type,
                    "document_index": int(doc_idx),
                    "document_id": doc_ids[doc_idx],
                    "qrel_score": int(relevance_map.get(doc_ids[doc_idx], 0)),
                    "exact_rank": exact_rank_map.get(doc_idx, np.nan),
                    "compressed_rank": compressed_rank_map.get(doc_idx, np.nan),
                    "exact_score": exact_score_map.get(doc_idx, np.nan),
                    "compressed_score": compressed_score_map.get(doc_idx, np.nan),
                    "exact_boundary_margin": exact_margin,
                    "compressed_boundary_margin": compressed_margin,
                })

        boundary_rows.append({
            "query_position": int(query_pos),
            "query_id": qid,
            "split": split,
            "backend": backend,
            "exact_rank10_score": float(exact_scores_row[TOPK_CUTOFF - 1]),
            "exact_rank11_score": float(exact_scores_row[TOPK_CUTOFF]),
            "exact_top10_boundary_margin": exact_margin,
            "compressed_rank10_score": float(
                compressed_scores_row[TOPK_CUTOFF - 1]
            ),
            "compressed_rank11_score": float(
                compressed_scores_row[TOPK_CUTOFF]
            ),
            "compressed_top10_boundary_margin": compressed_margin,
            "top10_overlap_with_exact": float(
                len(exact_top10_set & compressed_top10_set) / TOPK_CUTOFF
            ),
            "relevant_drop_count": int(len(relevant_drops)),
            "nonrelevant_intrusion_count": int(
                len(nonrelevant_intrusions)
            ),
            "relevant_recovery_count": int(len(relevant_recoveries)),
            "shared_relevant_count": int(len(shared_relevant)),
        })

events_df = pd.DataFrame(event_rows)
boundary_df = pd.DataFrame(boundary_rows)

events_df.to_csv(
    RANK_FLIP_DIR / "rank_flip_events.csv",
    index=False,
    encoding="utf-8-sig",
)
boundary_df.to_csv(
    RANK_FLIP_DIR / "query_boundary_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Calibration-only document-level risk labels.
calibration_events = events_df[
    events_df["split"] == "calibration"
].copy()

event_counts = (
    calibration_events
    .pivot_table(
        index=["backend", "document_index", "document_id"],
        columns="event_type",
        values="query_id",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

for column in [
    "relevant_drop",
    "nonrelevant_intrusion",
    "relevant_recovery",
    "shared_relevant",
]:
    if column not in event_counts.columns:
        event_counts[column] = 0

event_counts = event_counts.rename(columns={
    "relevant_drop": "relevant_drop_count",
    "nonrelevant_intrusion": "nonrelevant_intrusion_count",
    "relevant_recovery": "relevant_recovery_count",
    "shared_relevant": "shared_relevant_count",
})

event_context = (
    calibration_events
    .groupby(
        ["backend", "document_index", "document_id"],
        as_index=False,
    )
    .agg(
        mean_exact_rank_when_event=("exact_rank", "mean"),
        mean_exact_boundary_margin_when_event=(
            "exact_boundary_margin",
            "mean",
        ),
    )
)

event_counts = event_counts.merge(
    event_context,
    on=["backend", "document_index", "document_id"],
    how="left",
)

# Boundary exposure means appearing at ranks 8-12 in exact or compressed
# retrieval for a calibration query.
boundary_exposure_rows = []

for backend, (_, compressed_rankings) in {
    "ivfpq_m96_np16": (pq_scores, pq_rankings_top100),
    "opq_ivfpq_m96_np16": (opq_scores, opq_rankings_top100),
}.items():
    for query_pos, split in enumerate(query_splits):
        if split != "calibration":
            continue

        exposure_indices = set(
            int(x)
            for x in exact_rankings[
                query_pos,
                BOUNDARY_WINDOW_START - 1:BOUNDARY_WINDOW_END,
            ]
            if int(x) >= 0
        )
        exposure_indices.update(
            int(x)
            for x in compressed_rankings[
                query_pos,
                BOUNDARY_WINDOW_START - 1:BOUNDARY_WINDOW_END,
            ]
            if int(x) >= 0
        )

        for doc_idx in exposure_indices:
            boundary_exposure_rows.append({
                "backend": backend,
                "document_index": int(doc_idx),
                "document_id": doc_ids[doc_idx],
                "boundary_exposure_count": 1,
            })

boundary_exposure_df = pd.DataFrame(boundary_exposure_rows)

if not boundary_exposure_df.empty:
    boundary_exposure_df = (
        boundary_exposure_df
        .groupby(
            ["backend", "document_index", "document_id"],
            as_index=False,
        )["boundary_exposure_count"]
        .sum()
    )

top10_exposure_rows = []

for backend, (_, compressed_rankings) in {
    "ivfpq_m96_np16": (pq_scores, pq_rankings_top100),
    "opq_ivfpq_m96_np16": (opq_scores, opq_rankings_top100),
}.items():
    for query_pos, split in enumerate(query_splits):
        if split != "calibration":
            continue

        for doc_idx in exact_rankings[query_pos, :TOPK_CUTOFF]:
            doc_idx = int(doc_idx)
            if doc_idx >= 0:
                top10_exposure_rows.append({
                    "backend": backend,
                    "document_index": doc_idx,
                    "document_id": doc_ids[doc_idx],
                    "exact_top10_exposure_count": 1,
                    "compressed_top10_exposure_count": 0,
                })

        for doc_idx in compressed_rankings[query_pos, :TOPK_CUTOFF]:
            doc_idx = int(doc_idx)
            if doc_idx >= 0:
                top10_exposure_rows.append({
                    "backend": backend,
                    "document_index": doc_idx,
                    "document_id": doc_ids[doc_idx],
                    "exact_top10_exposure_count": 0,
                    "compressed_top10_exposure_count": 1,
                })

top10_exposure_df = (
    pd.DataFrame(top10_exposure_rows)
    .groupby(
        ["backend", "document_index", "document_id"],
        as_index=False,
    )[
        [
            "exact_top10_exposure_count",
            "compressed_top10_exposure_count",
        ]
    ]
    .sum()
)

risk_df = pd.DataFrame({
    "backend": np.repeat(
        ["ivfpq_m96_np16", "opq_ivfpq_m96_np16"],
        len(doc_ids),
    ),
    "document_index": list(range(len(doc_ids))) * 2,
    "document_id": doc_ids * 2,
})

risk_df = risk_df.merge(
    event_counts,
    on=["backend", "document_index", "document_id"],
    how="left",
)

risk_df = risk_df.merge(
    boundary_exposure_df,
    on=["backend", "document_index", "document_id"],
    how="left",
)

risk_df = risk_df.merge(
    top10_exposure_df,
    on=["backend", "document_index", "document_id"],
    how="left",
)

count_columns = [
    "relevant_drop_count",
    "nonrelevant_intrusion_count",
    "relevant_recovery_count",
    "shared_relevant_count",
    "boundary_exposure_count",
    "exact_top10_exposure_count",
    "compressed_top10_exposure_count",
]

for column in count_columns:
    if column not in risk_df.columns:
        risk_df[column] = 0
    risk_df[column] = risk_df[column].fillna(0).astype(int)

for column in [
    "mean_exact_rank_when_event",
    "mean_exact_boundary_margin_when_event",
]:
    if column not in risk_df.columns:
        risk_df[column] = np.nan

risk_df["risk_score"] = (
    3.0 * risk_df["relevant_drop_count"]
    + 1.0 * risk_df["nonrelevant_intrusion_count"]
    + 1.0 * risk_df["boundary_exposure_count"]
)

risk_df["risk_tier"] = pd.cut(
    risk_df["risk_score"],
    bins=[-0.1, 0.0, 1.0, 3.0, np.inf],
    labels=["tier_0", "tier_1", "tier_2", "tier_3"],
).astype(str)

risk_df = risk_df.sort_values(
    ["backend", "risk_score", "relevant_drop_count"],
    ascending=[True, False, False],
).reset_index(drop=True)

risk_df.to_csv(
    RANK_FLIP_DIR / "document_risk_calibration.csv",
    index=False,
    encoding="utf-8-sig",
)

event_summary = (
    events_df
    .groupby(["split", "backend", "event_type"])
    .size()
    .rename("count")
    .reset_index()
)

boundary_summary = (
    boundary_df
    .groupby(["split", "backend"], as_index=False)
    .agg(
        mean_top10_overlap=("top10_overlap_with_exact", "mean"),
        mean_relevant_drops=("relevant_drop_count", "mean"),
        mean_nonrelevant_intrusions=(
            "nonrelevant_intrusion_count",
            "mean",
        ),
        mean_exact_margin=("exact_top10_boundary_margin", "mean"),
        mean_compressed_margin=(
            "compressed_top10_boundary_margin",
            "mean",
        ),
    )
)

summary_lines = [
    "# FiQA PQ / OPQ Rank-Flip Summary",
    "",
    "## Event counts",
    "",
    event_summary.to_markdown(index=False),
    "",
    "## Query boundary summary",
    "",
    boundary_summary.to_markdown(index=False),
    "",
    "## Calibration risk labels",
    "",
    (
        "Risk labels use calibration queries only. "
        "risk = 3 * relevant_drop_count + "
        "1 * nonrelevant_intrusion_count + "
        "1 * boundary_exposure_count."
    ),
    "",
]

(RANK_FLIP_DIR / "rank_flip_summary.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("Saved rank-flip analysis outputs:")
for path in sorted(RANK_FLIP_DIR.iterdir()):
    print("-", path)

print("\nCalibration risk-label summary")
display(
    risk_df[
        risk_df["risk_score"] > 0
    ].groupby("backend", as_index=False).agg(
        documents_with_risk=("document_id", "count"),
        mean_risk=("risk_score", "mean"),
        max_risk=("risk_score", "max"),
        total_relevant_drops=("relevant_drop_count", "sum"),
    )
)

print("\nHeld-out rank-flip summary")
display(
    boundary_summary[
        boundary_summary["split"] == "heldout"
    ]
)


In [ ]:
# 9. Candidate-side recoverability audit
#
# Determines whether relevant documents dropped from compressed Top-10
# are still present in compressed Top-L candidate pools.
# Only documents still in the candidate pool can be recovered by a
# candidate-side residual refinement stage.

RECOVERABILITY_DEPTHS = [20, 50, 100]

relevant_drop_df = events_df[
    events_df["event_type"] == "relevant_drop"
].copy()

relevant_drop_df["compressed_rank"] = pd.to_numeric(
    relevant_drop_df["compressed_rank"],
    errors="coerce",
)

audit_rows = []

for (split, backend), group in relevant_drop_df.groupby(
    ["split", "backend"],
    dropna=False,
):
    total_drops = int(len(group))

    for depth in RECOVERABILITY_DEPTHS:
        recoverable_mask = (
            group["compressed_rank"].notna()
            & (group["compressed_rank"] <= depth)
        )

        recoverable_count = int(recoverable_mask.sum())
        unretrieved_count = int(
            group["compressed_rank"].isna().sum()
        )

        audit_rows.append({
            "split": split,
            "backend": backend,
            "candidate_depth": int(depth),
            "relevant_drop_events": total_drops,
            "recoverable_drop_events": recoverable_count,
            "recoverable_rate": (
                float(recoverable_count / total_drops)
                if total_drops else np.nan
            ),
            "not_in_top100_count": unretrieved_count,
            "not_in_top100_rate": (
                float(unretrieved_count / total_drops)
                if total_drops else np.nan
            ),
        })

recoverability_df = pd.DataFrame(audit_rows).sort_values(
    ["split", "backend", "candidate_depth"]
).reset_index(drop=True)

per_drop_df = relevant_drop_df[
    [
        "query_position",
        "query_id",
        "split",
        "backend",
        "document_index",
        "document_id",
        "qrel_score",
        "exact_rank",
        "compressed_rank",
        "exact_score",
        "compressed_score",
        "exact_boundary_margin",
        "compressed_boundary_margin",
    ]
].copy()

per_drop_df["recoverable_at_20"] = (
    per_drop_df["compressed_rank"].notna()
    & (per_drop_df["compressed_rank"] <= 20)
)
per_drop_df["recoverable_at_50"] = (
    per_drop_df["compressed_rank"].notna()
    & (per_drop_df["compressed_rank"] <= 50)
)
per_drop_df["recoverable_at_100"] = (
    per_drop_df["compressed_rank"].notna()
    & (per_drop_df["compressed_rank"] <= 100)
)

recoverability_df.to_csv(
    RANK_FLIP_DIR / "relevant_drop_recoverability_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

per_drop_df.to_csv(
    RANK_FLIP_DIR / "relevant_drop_recoverability_events.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# FiQA Candidate-Side Recoverability Audit",
    "",
    "A relevant drop is recoverable at depth L only when the dropped",
    "document still appears in the compressed Top-L candidate pool.",
    "",
    recoverability_df.to_markdown(index=False),
    "",
    "Interpretation:",
    "- Top-20 recoverability estimates the feasibility of a low-latency refinement stage.",
    "- Top-50 and Top-100 show the trade-off between candidate coverage and refinement cost.",
    "- Documents absent from Top-100 cannot be recovered by a Top-100-only refinement path.",
]

(RANK_FLIP_DIR / "recoverability_audit.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("Candidate-side recoverability audit")
display(recoverability_df)

print("\nHeld-out recoverability only")
display(
    recoverability_df[
        recoverability_df["split"] == "heldout"
    ]
)

print("\nSaved:")
print("-", RANK_FLIP_DIR / "relevant_drop_recoverability_summary.csv")
print("-", RANK_FLIP_DIR / "relevant_drop_recoverability_events.csv")
print("-", RANK_FLIP_DIR / "recoverability_audit.md")


In [ ]:
# 10. M=32 low-rate rank-flip audit for selective residual refinement
#
# Risk labels here are specific to the M=32 base index.
# Only calibration query labels are used for sidecar-selection signals.

M32_PQ_M = 32
M32_NBITS = 8
M32_EXPORT_K = 100
M32_TOPK = 10

M32_DIR = RESULT_DIR / "selective_residual_m32"
M32_DIR.mkdir(parents=True, exist_ok=True)

print("Training M=32 IVF-PQ base index...")
m32_index = faiss.GpuIndexIVFPQ(
    gpu_resources,
    D,
    FAISS_NLIST,
    M32_PQ_M,
    M32_NBITS,
    faiss.METRIC_INNER_PRODUCT,
    ivfpq_config,
)

m32_index.train(X_train_gpu)
m32_index.add(X_docs_gpu)
m32_index.nprobe = FAISS_NPROBE

print("Retrieving M=32 Top-100...")
m32_scores, m32_rankings_top100 = faiss_search_with_scores(
    m32_index,
    X_queries_gpu,
    M32_EXPORT_K,
)

m32_event_rows = []
m32_boundary_rows = []

for query_pos, (qid, split) in enumerate(zip(query_ids, query_splits)):
    relevance_map = qrels[qid]

    exact_ids = exact_rankings[query_pos]
    exact_scores_row = exact_scores[query_pos]
    m32_ids = m32_rankings_top100[query_pos]
    m32_scores_row = m32_scores[query_pos]

    exact_top10 = {
        int(idx)
        for idx in exact_ids[:M32_TOPK]
        if int(idx) >= 0
    }
    m32_top10 = {
        int(idx)
        for idx in m32_ids[:M32_TOPK]
        if int(idx) >= 0
    }

    exact_rank_map = {
        int(idx): rank
        for rank, idx in enumerate(exact_ids, start=1)
        if int(idx) >= 0
    }
    m32_rank_map = {
        int(idx): rank
        for rank, idx in enumerate(m32_ids, start=1)
        if int(idx) >= 0
    }

    exact_score_map = {
        int(idx): float(score)
        for idx, score in zip(exact_ids, exact_scores_row)
        if int(idx) >= 0
    }
    m32_score_map = {
        int(idx): float(score)
        for idx, score in zip(m32_ids, m32_scores_row)
        if int(idx) >= 0
    }

    exact_margin = float(
        exact_scores_row[M32_TOPK - 1] - exact_scores_row[M32_TOPK]
    )
    m32_margin = float(
        m32_scores_row[M32_TOPK - 1] - m32_scores_row[M32_TOPK]
    )

    exact_relevant = {
        idx for idx in exact_top10
        if relevance_map.get(doc_ids[idx], 0) > 0
    }
    m32_relevant = {
        idx for idx in m32_top10
        if relevance_map.get(doc_ids[idx], 0) > 0
    }

    relevant_drops = exact_relevant - m32_top10
    nonrelevant_intrusions = {
        idx for idx in (m32_top10 - exact_top10)
        if relevance_map.get(doc_ids[idx], 0) <= 0
    }
    relevant_recoveries = m32_relevant - exact_top10
    shared_relevant = exact_relevant & m32_relevant

    for event_type, indices in [
        ("relevant_drop", relevant_drops),
        ("nonrelevant_intrusion", nonrelevant_intrusions),
        ("relevant_recovery", relevant_recoveries),
        ("shared_relevant", shared_relevant),
    ]:
        for doc_idx in sorted(indices):
            m32_event_rows.append({
                "query_position": int(query_pos),
                "query_id": qid,
                "split": split,
                "backend": "ivfpq_m32_np16",
                "event_type": event_type,
                "document_index": int(doc_idx),
                "document_id": doc_ids[doc_idx],
                "qrel_score": int(relevance_map.get(doc_ids[doc_idx], 0)),
                "exact_rank": exact_rank_map.get(doc_idx, np.nan),
                "compressed_rank": m32_rank_map.get(doc_idx, np.nan),
                "exact_score": exact_score_map.get(doc_idx, np.nan),
                "compressed_score": m32_score_map.get(doc_idx, np.nan),
                "exact_boundary_margin": exact_margin,
                "compressed_boundary_margin": m32_margin,
            })

    m32_boundary_rows.append({
        "query_position": int(query_pos),
        "query_id": qid,
        "split": split,
        "backend": "ivfpq_m32_np16",
        "top10_overlap_with_exact": float(
            len(exact_top10 & m32_top10) / M32_TOPK
        ),
        "relevant_drop_count": int(len(relevant_drops)),
        "nonrelevant_intrusion_count": int(len(nonrelevant_intrusions)),
        "relevant_recovery_count": int(len(relevant_recoveries)),
        "exact_top10_boundary_margin": exact_margin,
        "compressed_top10_boundary_margin": m32_margin,
    })

m32_events_df = pd.DataFrame(m32_event_rows)
m32_boundary_df = pd.DataFrame(m32_boundary_rows)

m32_events_df.to_csv(
    M32_DIR / "m32_rank_flip_events.csv",
    index=False,
    encoding="utf-8-sig",
)
m32_boundary_df.to_csv(
    M32_DIR / "m32_query_boundary_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

cal_events = m32_events_df[
    m32_events_df["split"] == "calibration"
].copy()

m32_counts = (
    cal_events
    .pivot_table(
        index=["document_index", "document_id"],
        columns="event_type",
        values="query_id",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

for name in [
    "relevant_drop",
    "nonrelevant_intrusion",
    "relevant_recovery",
    "shared_relevant",
]:
    if name not in m32_counts.columns:
        m32_counts[name] = 0

m32_counts = m32_counts.rename(columns={
    "relevant_drop": "relevant_drop_count",
    "nonrelevant_intrusion": "nonrelevant_intrusion_count",
    "relevant_recovery": "relevant_recovery_count",
    "shared_relevant": "shared_relevant_count",
})

boundary_exposure_rows = []

for query_pos, split in enumerate(query_splits):
    if split != "calibration":
        continue

    candidate_ids = set(
        int(idx)
        for idx in exact_rankings[query_pos, 7:12]
        if int(idx) >= 0
    )
    candidate_ids.update(
        int(idx)
        for idx in m32_rankings_top100[query_pos, 7:12]
        if int(idx) >= 0
    )

    for doc_idx in candidate_ids:
        boundary_exposure_rows.append({
            "document_index": doc_idx,
            "document_id": doc_ids[doc_idx],
            "boundary_exposure_count": 1,
        })

m32_boundary_exposure_df = (
    pd.DataFrame(boundary_exposure_rows)
    .groupby(["document_index", "document_id"], as_index=False)
    ["boundary_exposure_count"]
    .sum()
)

m32_risk_df = pd.DataFrame({
    "document_index": np.arange(N_DOCS, dtype=np.int64),
    "document_id": doc_ids,
})

m32_risk_df = m32_risk_df.merge(
    m32_counts,
    on=["document_index", "document_id"],
    how="left",
).merge(
    m32_boundary_exposure_df,
    on=["document_index", "document_id"],
    how="left",
)

for column in [
    "relevant_drop_count",
    "nonrelevant_intrusion_count",
    "relevant_recovery_count",
    "shared_relevant_count",
    "boundary_exposure_count",
]:
    if column not in m32_risk_df.columns:
        m32_risk_df[column] = 0
    m32_risk_df[column] = m32_risk_df[column].fillna(0).astype(int)

m32_risk_df["drop_risk_score"] = (
    3.0 * m32_risk_df["relevant_drop_count"]
    + 1.0 * m32_risk_df["boundary_exposure_count"]
)

m32_risk_df["intrusion_risk_score"] = (
    1.0 * m32_risk_df["nonrelevant_intrusion_count"]
    + 1.0 * m32_risk_df["boundary_exposure_count"]
)

m32_risk_df = m32_risk_df.sort_values(
    ["drop_risk_score", "relevant_drop_count", "boundary_exposure_count"],
    ascending=False,
).reset_index(drop=True)

m32_risk_df.to_csv(
    M32_DIR / "m32_document_drop_risk_calibration.csv",
    index=False,
    encoding="utf-8-sig",
)

m32_metrics = per_query_metrics(
    m32_rankings_top100[:, :10],
    query_ids,
    doc_ids,
    qrels,
    FINAL_K,
)

m32_summary = pd.DataFrame([
    {
        "split": split,
        "query_count": int(mask.sum()),
        **aggregate_metrics(m32_metrics, mask),
        "mean_top10_overlap_with_exact": float(
            m32_boundary_df.loc[mask, "top10_overlap_with_exact"].mean()
        ),
        "mean_relevant_drops": float(
            m32_boundary_df.loc[mask, "relevant_drop_count"].mean()
        ),
    }
    for split, mask in {
        "all": np.ones(len(query_ids), dtype=bool),
        "calibration": calibration_mask,
        "heldout": heldout_mask,
    }.items()
])

m32_summary.to_csv(
    M32_DIR / "m32_quality_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("M=32 quality and instability summary")
display(m32_summary)

print("\nCalibration drop-risk distribution")
display(
    m32_risk_df[
        m32_risk_df["drop_risk_score"] > 0
    ].head(25)
)

print("\nSaved M=32 audit outputs:")
for path in sorted(M32_DIR.iterdir()):
    print("-", path)


In [ ]:
# 11. Fixed-budget selective residual refinement
#
# Compares calibration-risk allocation against random, reconstruction-error,
# and a storage-matched uniform M=48 IVF-PQ baseline.
#
# Cell 10 is required because it provides:
# - m32_index / m32_scores / m32_rankings_top100
# - m32_risk_df / m32_events_df
# - M32_DIR

import json
import time

BASE_PQ_BYTES = 32
EXTRA_BUDGET_BYTES_PER_VECTOR = 16
SIDECAR_VECTOR_BYTES = D * 2  # FP16 residual vector
SIDECAR_ID_BYTES = 4
SIDECAR_ENTRY_BYTES = SIDECAR_VECTOR_BYTES + SIDECAR_ID_BYTES

REFINEMENT_DEPTHS = [20, 50, 100]
RANDOM_SEED = 20260705

SELECTED_COUNT = int(
    np.floor(
        N_DOCS * EXTRA_BUDGET_BYTES_PER_VECTOR / SIDECAR_ENTRY_BYTES
    )
)

ACTUAL_EXTRA_BYTES_PER_VECTOR = (
    SELECTED_COUNT * SIDECAR_ENTRY_BYTES / N_DOCS
)
SELECTIVE_TOTAL_BYTES_PER_VECTOR = (
    BASE_PQ_BYTES + ACTUAL_EXTRA_BYTES_PER_VECTOR
)

print("Selective residual configuration")
print(f"Documents: {N_DOCS:,}")
print(f"Embedding dimension: {D}")
print(f"Selected sidecar vectors: {SELECTED_COUNT:,}")
print(f"Sidecar entry bytes: {SIDECAR_ENTRY_BYTES}")
print(f"Actual extra bytes/vector: {ACTUAL_EXTRA_BYTES_PER_VECTOR:.4f}")
print(
    "Selective total bytes/vector: "
    f"{SELECTIVE_TOTAL_BYTES_PER_VECTOR:.4f}"
)

# ------------------------------------------------------------------
# Decode the trained M=32 PQ approximation, then form x - x_hat.
# The residual payload is stored as FP16 only for selected documents.
# ------------------------------------------------------------------
print("\nDecoding M=32 IVF-PQ reconstructions...")
m32_cpu_index = faiss.index_gpu_to_cpu(m32_index)

try:
    m32_cpu_index.make_direct_map()
    m32_reconstructed_docs = m32_cpu_index.reconstruct_n(0, N_DOCS)
except Exception as exc:
    raise RuntimeError(
        "Unable to reconstruct M=32 vectors from the CPU IVF-PQ clone. "
        "This is required to form residual sidecars."
    ) from exc

m32_reconstructed_docs = np.ascontiguousarray(
    m32_reconstructed_docs,
    dtype=np.float32,
)
docs_cpu = np.ascontiguousarray(X_docs_gpu.detach().cpu().numpy(), dtype=np.float32)
queries_cpu = np.ascontiguousarray(X_queries_gpu.detach().cpu().numpy(), dtype=np.float32)

m32_residuals = np.ascontiguousarray(
    docs_cpu - m32_reconstructed_docs,
    dtype=np.float32,
)

reconstruction_error = np.sum(
    m32_residuals * m32_residuals,
    axis=1,
)

# ------------------------------------------------------------------
# Selection policies.
# Risk labels are calibration-only because m32_risk_df was built in Cell 10.
# ------------------------------------------------------------------
rng = np.random.default_rng(RANDOM_SEED)

random_selected = np.sort(
    rng.choice(
        N_DOCS,
        size=SELECTED_COUNT,
        replace=False,
    ).astype(np.int64)
)

reconstruction_error_selected = np.sort(
    np.argsort(-reconstruction_error, kind="stable")[:SELECTED_COUNT]
    .astype(np.int64)
)

risk_scores_by_doc = (
    m32_risk_df
    .set_index("document_index")["drop_risk_score"]
    .reindex(np.arange(N_DOCS), fill_value=0.0)
    .to_numpy(dtype=np.float64)
)

# Randomized deterministic tie-breaking avoids selecting arbitrary
# document-index ranges among documents with identical zero risk.
risk_tie_break = rng.permutation(N_DOCS)
risk_order = np.lexsort(
    (
        risk_tie_break,
        -risk_scores_by_doc,
    )
)

drop_risk_selected = np.sort(
    risk_order[:SELECTED_COUNT].astype(np.int64)
)

selection_sets = {
    "random_sidecar": random_selected,
    "reconstruction_error_sidecar": reconstruction_error_selected,
    "calibration_drop_risk_sidecar": drop_risk_selected,
}

selection_rows = []

for policy, selected_ids in selection_sets.items():
    for rank, doc_idx in enumerate(selected_ids, start=1):
        selection_rows.append({
            "policy": policy,
            "selection_rank": int(rank),
            "document_index": int(doc_idx),
            "document_id": doc_ids[doc_idx],
            "drop_risk_score": float(risk_scores_by_doc[doc_idx]),
            "reconstruction_error_l2_sq": float(
                reconstruction_error[doc_idx]
            ),
        })

selection_df = pd.DataFrame(selection_rows)

selection_df.to_csv(
    M32_DIR / "selective_residual_selected_documents.csv",
    index=False,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------------
# Candidate-side residual correction.
#
# Only candidates already present in compressed Top-L can be corrected.
# Each selected vector stores x - x_hat as FP16.
# ------------------------------------------------------------------
def refine_candidates_with_residuals(
    candidate_ids,
    candidate_scores,
    selected_doc_ids,
    depth,
):
    selected_residuals_fp16 = m32_residuals[selected_doc_ids].astype(
        np.float16
    )
    residual_lookup = {
        int(doc_idx): selected_residuals_fp16[pos]
        for pos, doc_idx in enumerate(selected_doc_ids)
    }

    refined_rankings = np.full(
        (candidate_ids.shape[0], FINAL_K),
        -1,
        dtype=np.int64,
    )

    refined_count = 0
    start = time.perf_counter()

    for query_pos in range(candidate_ids.shape[0]):
        ids = candidate_ids[query_pos, :depth].astype(
            np.int64,
            copy=True,
        )
        scores = candidate_scores[query_pos, :depth].astype(
            np.float32,
            copy=True,
        )
        query_vector = queries_cpu[query_pos]

        for candidate_pos, doc_idx in enumerate(ids):
            residual_fp16 = residual_lookup.get(int(doc_idx))

            if residual_fp16 is None:
                continue

            scores[candidate_pos] += float(
                np.dot(
                    query_vector,
                    residual_fp16.astype(np.float32),
                )
            )
            refined_count += 1

        order = np.argsort(-scores, kind="stable")
        refined_rankings[query_pos] = ids[order[:FINAL_K]]

    elapsed_seconds = time.perf_counter() - start

    return {
        "rankings": refined_rankings,
        "latency_ms_per_query": (
            elapsed_seconds * 1000.0 / candidate_ids.shape[0]
        ),
        "refined_candidate_occurrences": int(refined_count),
    }


def mean_top10_overlap_with_exact(rankings, mask):
    overlaps = []

    for query_pos in np.flatnonzero(mask):
        exact_top10 = {
            int(doc_idx)
            for doc_idx in exact_rankings[query_pos, :FINAL_K]
            if int(doc_idx) >= 0
        }
        predicted_top10 = {
            int(doc_idx)
            for doc_idx in rankings[query_pos, :FINAL_K]
            if int(doc_idx) >= 0
        }

        overlaps.append(
            len(exact_top10 & predicted_top10) / FINAL_K
        )

    return float(np.mean(overlaps))


heldout_drop_lookup = {}

for row in m32_events_df[
    (m32_events_df["split"] == "heldout")
    & (m32_events_df["event_type"] == "relevant_drop")
].itertuples(index=False):
    heldout_drop_lookup.setdefault(
        int(row.query_position),
        set(),
    ).add(int(row.document_index))


def relevant_drop_recovery_rate(rankings):
    total_drops = 0
    recovered_drops = 0

    for query_pos, dropped_docs in heldout_drop_lookup.items():
        predicted_top10 = {
            int(doc_idx)
            for doc_idx in rankings[query_pos, :FINAL_K]
            if int(doc_idx) >= 0
        }

        total_drops += len(dropped_docs)
        recovered_drops += len(dropped_docs & predicted_top10)

    return {
        "heldout_relevant_drop_events": int(total_drops),
        "heldout_recovered_drop_events": int(recovered_drops),
        "heldout_relevant_drop_recovery_rate": (
            float(recovered_drops / total_drops)
            if total_drops else np.nan
        ),
    }


def heldout_result_row(
    method,
    policy,
    candidate_depth,
    rankings,
    refinement_latency_ms_per_query,
    refined_candidate_occurrences,
    total_bytes_per_vector,
    selected_document_count,
):
    metrics = per_query_metrics(
        rankings,
        query_ids,
        doc_ids,
        qrels,
        FINAL_K,
    )

    row = {
        "method": method,
        "policy": policy,
        "candidate_depth": candidate_depth,
        "selected_document_count": int(selected_document_count),
        "total_bytes_per_vector": float(total_bytes_per_vector),
        "refinement_latency_ms_per_query": float(
            refinement_latency_ms_per_query
        ),
        "refined_candidate_occurrences": int(
            refined_candidate_occurrences
        ),
        "mean_top10_overlap_with_exact": mean_top10_overlap_with_exact(
            rankings,
            heldout_mask,
        ),
    }

    row.update(aggregate_metrics(metrics, heldout_mask))
    row.update(relevant_drop_recovery_rate(rankings))

    return row


result_rows = []

# Base M=32 reference: no sidecar, no refinement.
base_m32_rankings = m32_rankings_top100[:, :FINAL_K].astype(
    np.int64,
    copy=True,
)

result_rows.append(
    heldout_result_row(
        method="base_ivfpq_m32",
        policy="none",
        candidate_depth=0,
        rankings=base_m32_rankings,
        refinement_latency_ms_per_query=0.0,
        refined_candidate_occurrences=0,
        total_bytes_per_vector=float(BASE_PQ_BYTES),
        selected_document_count=0,
    )
)

# Selective residual policies at Top-20 / Top-50 / Top-100.
for policy, selected_ids in selection_sets.items():
    for depth in REFINEMENT_DEPTHS:
        refined = refine_candidates_with_residuals(
            candidate_ids=m32_rankings_top100,
            candidate_scores=m32_scores,
            selected_doc_ids=selected_ids,
            depth=depth,
        )

        result_rows.append(
            heldout_result_row(
                method="selective_residual_m32",
                policy=policy,
                candidate_depth=int(depth),
                rankings=refined["rankings"],
                refinement_latency_ms_per_query=(
                    refined["latency_ms_per_query"]
                ),
                refined_candidate_occurrences=(
                    refined["refined_candidate_occurrences"]
                ),
                total_bytes_per_vector=(
                    SELECTIVE_TOTAL_BYTES_PER_VECTOR
                ),
                selected_document_count=SELECTED_COUNT,
            )
        )

# ------------------------------------------------------------------
# Storage-matched uniform M=48 baseline.
# ------------------------------------------------------------------
print("\nTraining storage-matched uniform M=48 IVF-PQ baseline...")

m48_index = faiss.GpuIndexIVFPQ(
    gpu_resources,
    D,
    FAISS_NLIST,
    48,
    8,
    faiss.METRIC_INNER_PRODUCT,
    ivfpq_config,
)

m48_index.train(X_train_gpu)
m48_index.add(X_docs_gpu)
m48_index.nprobe = FAISS_NPROBE

m48_start = time.perf_counter()

m48_scores, m48_rankings_top100 = faiss_search_with_scores(
    m48_index,
    X_queries_gpu,
    100,
)

m48_elapsed_seconds = time.perf_counter() - m48_start

result_rows.append(
    heldout_result_row(
        method="uniform_ivfpq_m48",
        policy="uniform_code_budget",
        candidate_depth=100,
        rankings=m48_rankings_top100[:, :FINAL_K],
        refinement_latency_ms_per_query=(
            m48_elapsed_seconds * 1000.0 / len(query_ids)
        ),
        refined_candidate_occurrences=0,
        total_bytes_per_vector=48.0,
        selected_document_count=N_DOCS,
    )
)

results_df = pd.DataFrame(result_rows).sort_values(
    [
        "method",
        "policy",
        "candidate_depth",
    ]
).reset_index(drop=True)

results_df.to_csv(
    M32_DIR / "selective_residual_heldout_results.csv",
    index=False,
    encoding="utf-8-sig",
)

config = {
    "base_pq_m": 32,
    "uniform_baseline_pq_m": 48,
    "nlist": int(FAISS_NLIST),
    "nprobe": int(FAISS_NPROBE),
    "final_k": int(FINAL_K),
    "refinement_depths": REFINEMENT_DEPTHS,
    "sidecar_entry_bytes": int(SIDECAR_ENTRY_BYTES),
    "selected_document_count": int(SELECTED_COUNT),
    "actual_extra_bytes_per_vector": float(
        ACTUAL_EXTRA_BYTES_PER_VECTOR
    ),
    "selective_total_bytes_per_vector": float(
        SELECTIVE_TOTAL_BYTES_PER_VECTOR
    ),
    "random_seed": int(RANDOM_SEED),
    "risk_source": "calibration-only M=32 relevant-drop risk",
}

(M32_DIR / "selective_residual_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nHeld-out selective residual results")
display(results_df)

print("\nSaved:")
for filename in [
    "selective_residual_selected_documents.csv",
    "selective_residual_heldout_results.csv",
    "selective_residual_config.json",
]:
    print("-", M32_DIR / filename)


In [ ]:
# 12. Drop coverage and failure audit
#
# Diagnoses why held-out M=32 relevant drops were or were not recovered
# by the calibration-only drop-risk sidecar. This cell does not change
# sidecar selection or use held-out labels for allocation.

AUDIT_POLICY = "calibration_drop_risk_sidecar"
AUDIT_DEPTHS = [20, 50, 100]

audit_selected_ids = selection_sets[AUDIT_POLICY]
audit_selected_set = {
    int(doc_idx)
    for doc_idx in audit_selected_ids
}

audit_residual_lookup = {
    int(doc_idx): m32_residuals[pos].astype(np.float16)
    for pos, doc_idx in enumerate(audit_selected_ids)
}

heldout_drop_events_df = m32_events_df[
    (m32_events_df["split"] == "heldout")
    & (m32_events_df["event_type"] == "relevant_drop")
].copy()

if heldout_drop_events_df.empty:
    raise RuntimeError("No held-out relevant-drop events found.")

audit_rows = []

for depth in AUDIT_DEPTHS:
    corrected_rank_maps = {}

    for query_pos in sorted(
        heldout_drop_events_df["query_position"].unique()
    ):
        candidate_ids = m32_rankings_top100[query_pos, :depth].astype(
            np.int64,
            copy=True,
        )
        candidate_scores = m32_scores[query_pos, :depth].astype(
            np.float32,
            copy=True,
        )
        query_vector = queries_cpu[query_pos]

        for candidate_pos, doc_idx in enumerate(candidate_ids):
            residual_fp16 = audit_residual_lookup.get(int(doc_idx))

            if residual_fp16 is None:
                continue

            candidate_scores[candidate_pos] += float(
                np.dot(
                    query_vector,
                    residual_fp16.astype(np.float32),
                )
            )

        order = np.argsort(-candidate_scores, kind="stable")

        corrected_rank_maps[int(query_pos)] = {
            int(candidate_ids[position]): int(rank)
            for rank, position in enumerate(order, start=1)
        }

    for row in heldout_drop_events_df.itertuples(index=False):
        query_pos = int(row.query_position)
        doc_idx = int(row.document_index)

        compressed_rank = (
            int(row.compressed_rank)
            if pd.notna(row.compressed_rank)
            else None
        )

        in_candidate_pool = (
            compressed_rank is not None
            and compressed_rank <= depth
        )
        in_sidecar = doc_idx in audit_selected_set

        corrected_rank = corrected_rank_maps[query_pos].get(doc_idx)
        recovered_to_top10 = (
            corrected_rank is not None
            and corrected_rank <= FINAL_K
        )

        if not in_candidate_pool:
            failure_stage = "absent_from_candidate_topL"
        elif not in_sidecar:
            failure_stage = "candidate_not_selected_for_sidecar"
        elif recovered_to_top10:
            failure_stage = "recovered_to_top10"
        else:
            failure_stage = "corrected_but_not_recovered"

        audit_rows.append({
            "policy": AUDIT_POLICY,
            "candidate_depth": int(depth),
            "query_position": query_pos,
            "query_id": row.query_id,
            "document_index": doc_idx,
            "document_id": row.document_id,
            "qrel_score": int(row.qrel_score),
            "exact_rank": int(row.exact_rank),
            "compressed_rank": compressed_rank,
            "corrected_rank": corrected_rank,
            "in_candidate_pool": bool(in_candidate_pool),
            "in_sidecar": bool(in_sidecar),
            "recovered_to_top10": bool(recovered_to_top10),
            "failure_stage": failure_stage,
            "drop_risk_score": float(risk_scores_by_doc[doc_idx]),
            "reconstruction_error_l2_sq": float(
                reconstruction_error[doc_idx]
            ),
        })

failure_events_df = pd.DataFrame(audit_rows)

failure_summary_df = (
    failure_events_df
    .groupby(
        ["policy", "candidate_depth", "failure_stage"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "relevant_drop_events"})
)

failure_totals_df = (
    failure_events_df
    .groupby(
        ["policy", "candidate_depth"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "total_relevant_drop_events"})
)

failure_summary_df = failure_summary_df.merge(
    failure_totals_df,
    on=["policy", "candidate_depth"],
    how="left",
)

failure_summary_df["event_rate"] = (
    failure_summary_df["relevant_drop_events"]
    / failure_summary_df["total_relevant_drop_events"]
)

coverage_summary_df = (
    failure_events_df
    .groupby(
        ["policy", "candidate_depth"],
        as_index=False,
    )
    .agg(
        total_relevant_drop_events=(
            "document_index",
            "size",
        ),
        candidate_pool_covered_events=(
            "in_candidate_pool",
            "sum",
        ),
        selected_sidecar_events=(
            "in_sidecar",
            "sum",
        ),
        recovered_to_top10_events=(
            "recovered_to_top10",
            "sum",
        ),
        mean_drop_risk_score=(
            "drop_risk_score",
            "mean",
        ),
    )
)

coverage_summary_df[
    "candidate_pool_coverage_rate"
] = (
    coverage_summary_df["candidate_pool_covered_events"]
    / coverage_summary_df["total_relevant_drop_events"]
)

coverage_summary_df[
    "sidecar_selection_rate"
] = (
    coverage_summary_df["selected_sidecar_events"]
    / coverage_summary_df["total_relevant_drop_events"]
)

coverage_summary_df[
    "recovery_rate"
] = (
    coverage_summary_df["recovered_to_top10_events"]
    / coverage_summary_df["total_relevant_drop_events"]
)

failure_events_df.to_csv(
    M32_DIR / "selective_residual_drop_failure_events.csv",
    index=False,
    encoding="utf-8-sig",
)

failure_summary_df.to_csv(
    M32_DIR / "selective_residual_drop_failure_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

coverage_summary_df.to_csv(
    M32_DIR / "selective_residual_drop_coverage_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# Selective Residual Drop Coverage and Failure Audit",
    "",
    "Policy: calibration-only M=32 drop-risk sidecar.",
    "",
    "## Coverage summary",
    "",
    coverage_summary_df.to_markdown(index=False),
    "",
    "## Failure-stage summary",
    "",
    failure_summary_df.to_markdown(index=False),
    "",
    "Interpretation:",
    "- absent_from_candidate_topL: cannot be recovered by candidate-side refinement.",
    "- candidate_not_selected_for_sidecar: allocation coverage failure.",
    "- corrected_but_not_recovered: residual correction was insufficient to restore Top-10.",
    "- recovered_to_top10: successful candidate-side recovery.",
]

(M32_DIR / "selective_residual_drop_failure_audit.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("Held-out drop coverage summary")
display(coverage_summary_df)

print("\nHeld-out drop failure-stage summary")
display(
    failure_summary_df.sort_values(
        ["candidate_depth", "failure_stage"]
    )
)

print("\nSaved:")
for filename in [
    "selective_residual_drop_failure_events.csv",
    "selective_residual_drop_failure_summary.csv",
    "selective_residual_drop_coverage_summary.csv",
    "selective_residual_drop_failure_audit.md",
]:
    print("-", M32_DIR / filename)


In [ ]:
# 13. Calibration candidate-distortion sidecar allocation
#
# A label-free allocation policy derived only from calibration queries.
# It favors documents that:
# 1. appear in compressed M=32 candidate pools,
# 2. lie near the Top-10 decision boundary, and
# 3. have a large positive score correction available from residuals.
#
# No qrels are used in sidecar selection.

DISTORTION_POLICY = "calibration_candidate_distortion_sidecar"
DISTORTION_BOUNDARY_RANK = FINAL_K
DISTORTION_EXPORT_K = 100
DISTORTION_SEED = RANDOM_SEED + 13

candidate_frequency = np.zeros(N_DOCS, dtype=np.int32)
boundary_frequency = np.zeros(N_DOCS, dtype=np.int32)
positive_correction_sum = np.zeros(N_DOCS, dtype=np.float64)
weighted_distortion_sum = np.zeros(N_DOCS, dtype=np.float64)

calibration_positions = np.flatnonzero(calibration_mask)

for query_pos in calibration_positions:
    candidate_ids = m32_rankings_top100[
        query_pos,
        :DISTORTION_EXPORT_K,
    ].astype(np.int64)

    candidate_scores = m32_scores[
        query_pos,
        :DISTORTION_EXPORT_K,
    ].astype(np.float32)

    valid_mask = candidate_ids >= 0
    candidate_ids = candidate_ids[valid_mask]
    candidate_scores = candidate_scores[valid_mask]

    if len(candidate_ids) == 0:
        continue

    query_vector = queries_cpu[query_pos]

    # Exact dense dot product only for compressed candidates.
    exact_candidate_scores = (
        docs_cpu[candidate_ids] @ query_vector
    ).astype(np.float32)

    # Positive value means PQ underestimated this candidate's score.
    positive_corrections = np.maximum(
        exact_candidate_scores - candidate_scores,
        0.0,
    )

    ranks = np.arange(1, len(candidate_ids) + 1, dtype=np.int32)

    # Highest weight around ranks 10 and 11, then decays outward.
    boundary_weights = 1.0 / (
        1.0 + np.abs(ranks - DISTORTION_BOUNDARY_RANK)
    )

    candidate_frequency[candidate_ids] += 1
    positive_correction_sum[candidate_ids] += positive_corrections
    weighted_distortion_sum[candidate_ids] += (
        positive_corrections * boundary_weights
    )

    boundary_ids = candidate_ids[
        (ranks >= DISTORTION_BOUNDARY_RANK - 2)
        & (ranks <= DISTORTION_BOUNDARY_RANK + 2)
    ]
    boundary_frequency[boundary_ids] += 1

# Sum already rewards repeated candidate exposure. The second term gives
# an explicit small preference to documents repeatedly near the boundary.
candidate_distortion_score = (
    weighted_distortion_sum
    + 0.01 * boundary_frequency.astype(np.float64)
)

distortion_tie_rng = np.random.default_rng(DISTORTION_SEED)
distortion_tie_break = distortion_tie_rng.permutation(N_DOCS)

distortion_order = np.lexsort(
    (
        distortion_tie_break,
        -candidate_frequency,
        -candidate_distortion_score,
    )
)

candidate_distortion_selected = np.sort(
    distortion_order[:SELECTED_COUNT].astype(np.int64)
)

candidate_distortion_df = pd.DataFrame({
    "document_index": np.arange(N_DOCS, dtype=np.int64),
    "document_id": doc_ids,
    "candidate_frequency_calibration": candidate_frequency,
    "boundary_frequency_calibration": boundary_frequency,
    "positive_score_correction_sum": positive_correction_sum,
    "weighted_distortion_sum": weighted_distortion_sum,
    "candidate_distortion_score": candidate_distortion_score,
    "drop_risk_score": risk_scores_by_doc,
    "reconstruction_error_l2_sq": reconstruction_error,
})

candidate_distortion_df["selected_for_sidecar"] = False
candidate_distortion_df.loc[
    candidate_distortion_selected,
    "selected_for_sidecar",
] = True

candidate_distortion_df = candidate_distortion_df.sort_values(
    [
        "selected_for_sidecar",
        "candidate_distortion_score",
        "candidate_frequency_calibration",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

candidate_distortion_df.to_csv(
    M32_DIR / "candidate_distortion_allocation_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

candidate_distortion_selected_df = candidate_distortion_df[
    candidate_distortion_df["selected_for_sidecar"]
].copy()

candidate_distortion_selected_df.insert(
    0,
    "selection_rank",
    np.arange(1, len(candidate_distortion_selected_df) + 1),
)

candidate_distortion_selected_df.to_csv(
    M32_DIR / "candidate_distortion_selected_documents.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Candidate-distortion allocation summary")
print(f"Selected documents: {len(candidate_distortion_selected):,}")
print(
    "Selected docs with positive distortion score: "
    f"{int((candidate_distortion_selected_df['candidate_distortion_score'] > 0).sum()):,}"
)
print(
    "Selected docs observed in calibration Top-100: "
    f"{int((candidate_distortion_selected_df['candidate_frequency_calibration'] > 0).sum()):,}"
)

display(
    candidate_distortion_selected_df[
        [
            "selection_rank",
            "document_id",
            "candidate_frequency_calibration",
            "boundary_frequency_calibration",
            "positive_score_correction_sum",
            "weighted_distortion_sum",
            "candidate_distortion_score",
            "drop_risk_score",
            "reconstruction_error_l2_sq",
        ]
    ].head(25)
)

candidate_distortion_result_rows = []

for depth in REFINEMENT_DEPTHS:
    refined = refine_candidates_with_residuals(
        candidate_ids=m32_rankings_top100,
        candidate_scores=m32_scores,
        selected_doc_ids=candidate_distortion_selected,
        depth=depth,
    )

    candidate_distortion_result_rows.append(
        heldout_result_row(
            method="selective_residual_m32",
            policy=DISTORTION_POLICY,
            candidate_depth=int(depth),
            rankings=refined["rankings"],
            refinement_latency_ms_per_query=(
                refined["latency_ms_per_query"]
            ),
            refined_candidate_occurrences=(
                refined["refined_candidate_occurrences"]
            ),
            total_bytes_per_vector=(
                SELECTIVE_TOTAL_BYTES_PER_VECTOR
            ),
            selected_document_count=SELECTED_COUNT,
        )
    )

candidate_distortion_results_df = pd.DataFrame(
    candidate_distortion_result_rows
).sort_values("candidate_depth").reset_index(drop=True)

candidate_distortion_results_df.to_csv(
    M32_DIR / "candidate_distortion_heldout_results.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison_columns = [
    "method",
    "policy",
    "candidate_depth",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
    "mean_top10_overlap_with_exact",
    "heldout_recovered_drop_events",
    "heldout_relevant_drop_recovery_rate",
    "refined_candidate_occurrences",
    "refinement_latency_ms_per_query",
    "total_bytes_per_vector",
]

comparison_df = pd.concat(
    [
        results_df[
            results_df["policy"].isin(
                [
                    "random_sidecar",
                    "reconstruction_error_sidecar",
                    "calibration_drop_risk_sidecar",
                    "uniform_code_budget",
                    "none",
                ]
            )
        ],
        candidate_distortion_results_df,
    ],
    ignore_index=True,
).sort_values(
    ["method", "policy", "candidate_depth"]
).reset_index(drop=True)

comparison_df.to_csv(
    M32_DIR / "selective_residual_policy_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# Calibration Candidate-Distortion Sidecar Allocation",
    "",
    "Selection uses calibration M=32 candidate behavior only.",
    "No relevance labels are used in this policy.",
    "",
    "## Held-out result",
    "",
    candidate_distortion_results_df[
        comparison_columns
    ].to_markdown(index=False),
    "",
    "## Selected-sidecar diagnostics",
    "",
    candidate_distortion_selected_df[
        [
            "candidate_frequency_calibration",
            "boundary_frequency_calibration",
            "positive_score_correction_sum",
            "weighted_distortion_sum",
            "candidate_distortion_score",
            "drop_risk_score",
            "reconstruction_error_l2_sq",
        ]
    ].describe().to_markdown(),
]

(M32_DIR / "candidate_distortion_allocation_audit.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("\nHeld-out candidate-distortion results")
display(candidate_distortion_results_df)

print("\nSaved:")
for filename in [
    "candidate_distortion_allocation_scores.csv",
    "candidate_distortion_selected_documents.csv",
    "candidate_distortion_heldout_results.csv",
    "selective_residual_policy_comparison.csv",
    "candidate_distortion_allocation_audit.md",
]:
    print("-", M32_DIR / filename)


In [ ]:
# 14. Oracle exact candidate rescoring ceiling
#
# This is an upper bound for candidate-side refinement:
# within the compressed M=32 Top-L candidate pool, replace approximate
# PQ scores with exact Float32 inner products against original documents.
#
# It does not represent deployable storage or latency. It answers whether
# a compressed residual sidecar could plausibly have enough headroom.

ORACLE_DEPTHS = [20, 50, 100]

oracle_rows = []
oracle_rankings_by_depth = {}

for depth in ORACLE_DEPTHS:
    oracle_rankings = np.full(
        (len(query_ids), FINAL_K),
        -1,
        dtype=np.int64,
    )

    start = time.perf_counter()

    for query_pos in range(len(query_ids)):
        candidate_ids = m32_rankings_top100[
            query_pos,
            :depth,
        ].astype(np.int64)

        valid_mask = candidate_ids >= 0
        candidate_ids = candidate_ids[valid_mask]

        exact_candidate_scores = (
            docs_cpu[candidate_ids] @ queries_cpu[query_pos]
        ).astype(np.float32)

        order = np.argsort(-exact_candidate_scores, kind="stable")

        oracle_rankings[
            query_pos,
            :min(FINAL_K, len(candidate_ids))
        ] = candidate_ids[order[:FINAL_K]]

    elapsed_seconds = time.perf_counter() - start
    oracle_rankings_by_depth[depth] = oracle_rankings

    oracle_rows.append(
        heldout_result_row(
            method="oracle_exact_candidate_rescore",
            policy="all_candidates_exact_fp32",
            candidate_depth=int(depth),
            rankings=oracle_rankings,
            refinement_latency_ms_per_query=(
                elapsed_seconds * 1000.0 / len(query_ids)
            ),
            refined_candidate_occurrences=int(
                len(query_ids) * depth
            ),
            total_bytes_per_vector=np.nan,
            selected_document_count=N_DOCS,
        )
    )

oracle_results_df = pd.DataFrame(oracle_rows).sort_values(
    "candidate_depth"
).reset_index(drop=True)

oracle_results_df.to_csv(
    M32_DIR / "oracle_candidate_rescoring_heldout_results.csv",
    index=False,
    encoding="utf-8-sig",
)

base_and_uniform_df = results_df[
    results_df["policy"].isin(
        ["none", "uniform_code_budget"]
    )
].copy()

oracle_comparison_df = pd.concat(
    [
        base_and_uniform_df,
        candidate_distortion_results_df,
        oracle_results_df,
    ],
    ignore_index=True,
).sort_values(
    ["method", "policy", "candidate_depth"]
).reset_index(drop=True)

oracle_comparison_df.to_csv(
    M32_DIR / "oracle_candidate_rescoring_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

display_columns = [
    "method",
    "policy",
    "candidate_depth",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
    "mean_top10_overlap_with_exact",
    "heldout_recovered_drop_events",
    "heldout_relevant_drop_recovery_rate",
    "refinement_latency_ms_per_query",
]

summary_lines = [
    "# Oracle Exact Candidate Rescoring Ceiling",
    "",
    "Within each M=32 compressed Top-L candidate pool, all candidates are",
    "rescored with exact Float32 document-query inner products.",
    "",
    "This is an upper bound for candidate-side refinement, not a deployable",
    "method or a storage-matched comparison.",
    "",
    "## Held-out oracle result",
    "",
    oracle_results_df[display_columns].to_markdown(index=False),
    "",
    "## Comparison with base, candidate-distortion, and uniform M=48",
    "",
    oracle_comparison_df[display_columns].to_markdown(index=False),
    "",
    "Interpretation:",
    "- If oracle Top-L remains far below uniform M=48, candidate-side residual refinement has limited headroom.",
    "- If oracle Top-L approaches or exceeds uniform M=48, a more storage-efficient residual sidecar remains worth pursuing.",
    "- Oracle cannot recover documents absent from the compressed Top-L pool.",
]

(M32_DIR / "oracle_candidate_rescoring_ceiling.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("Held-out oracle candidate-rescoring ceiling")
display(oracle_results_df[display_columns])

print("\nComparison")
display(oracle_comparison_df[display_columns])

print("\nSaved:")
for filename in [
    "oracle_candidate_rescoring_heldout_results.csv",
    "oracle_candidate_rescoring_comparison.csv",
    "oracle_candidate_rescoring_ceiling.md",
]:
    print("-", M32_DIR / filename)


In [ ]:
# 15. Residual-PQ diagnostic code-size sweep
#
# Fixed selected set:
# candidate-distortion sidecar allocation from Cell 13.
#
# This is a diagnostic sweep, not a fixed-budget comparison.
# It measures how accurately compressed residual codes can approximate
# the FP16 residual correction for the same selected documents.

RESIDUAL_PQ_M_VALUES = [4, 8, 16, 32]
RESIDUAL_PQ_DEPTHS = [20, 50]
RESIDUAL_PQ_NBITS = 8
RESIDUAL_PQ_TRAIN_SAMPLES = min(20000, N_DOCS)
RESIDUAL_PQ_SEED = RANDOM_SEED + 15

RESIDUAL_PQ_DIR = M32_DIR / "residual_pq_sweep"
RESIDUAL_PQ_DIR.mkdir(parents=True, exist_ok=True)

selected_doc_ids = np.sort(
    candidate_distortion_selected.astype(np.int64)
)

selected_residuals = np.ascontiguousarray(
    m32_residuals[selected_doc_ids],
    dtype=np.float32,
)

rng = np.random.default_rng(RESIDUAL_PQ_SEED)
train_ids = np.sort(
    rng.choice(
        N_DOCS,
        size=RESIDUAL_PQ_TRAIN_SAMPLES,
        replace=False,
    ).astype(np.int64)
)

residual_train_vectors = np.ascontiguousarray(
    m32_residuals[train_ids],
    dtype=np.float32,
)

RESIDUAL_PQ_CODEBOOK_BYTES = D * (2 ** RESIDUAL_PQ_NBITS) * 4
RESIDUAL_PQ_CODEBOOK_AVG_BYTES = (
    RESIDUAL_PQ_CODEBOOK_BYTES / N_DOCS
)

print("Residual-PQ diagnostic sweep")
print(f"Selected sidecar documents: {len(selected_doc_ids):,}")
print(f"Residual PQ train vectors: {len(train_ids):,}")
print(f"Residual codebook bytes: {RESIDUAL_PQ_CODEBOOK_BYTES:,}")
print(
    "Amortized residual-PQ codebook bytes/vector: "
    f"{RESIDUAL_PQ_CODEBOOK_AVG_BYTES:.4f}"
)

def refine_candidates_with_decoded_residuals(
    candidate_ids,
    candidate_scores,
    selected_ids,
    decoded_residuals,
    depth,
):
    residual_lookup = {
        int(doc_idx): decoded_residuals[pos]
        for pos, doc_idx in enumerate(selected_ids)
    }

    refined_rankings = np.full(
        (candidate_ids.shape[0], FINAL_K),
        -1,
        dtype=np.int64,
    )

    corrected_occurrences = 0
    start = time.perf_counter()

    for query_pos in range(candidate_ids.shape[0]):
        ids = candidate_ids[query_pos, :depth].astype(
            np.int64,
            copy=True,
        )
        scores = candidate_scores[query_pos, :depth].astype(
            np.float32,
            copy=True,
        )
        query_vector = queries_cpu[query_pos]

        for candidate_pos, doc_idx in enumerate(ids):
            residual = residual_lookup.get(int(doc_idx))
            if residual is None:
                continue

            scores[candidate_pos] += float(
                np.dot(query_vector, residual)
            )
            corrected_occurrences += 1

        order = np.argsort(-scores, kind="stable")
        refined_rankings[query_pos] = ids[order[:FINAL_K]]

    elapsed_seconds = time.perf_counter() - start

    return {
        "rankings": refined_rankings,
        "corrected_occurrences": int(corrected_occurrences),
        "latency_ms_per_query": (
            elapsed_seconds * 1000.0 / candidate_ids.shape[0]
        ),
    }


sweep_rows = []
selection_rows = []

for residual_m in RESIDUAL_PQ_M_VALUES:
    print(f"\nTraining residual PQ: M_r={residual_m}")

    residual_pq = faiss.ProductQuantizer(
        D,
        residual_m,
        RESIDUAL_PQ_NBITS,
    )

    try:
        residual_pq.cp.niter = 15
        residual_pq.cp.seed = RESIDUAL_PQ_SEED + residual_m
    except Exception:
        pass

    residual_pq.train(residual_train_vectors)

    residual_codes = residual_pq.compute_codes(selected_residuals)
    decoded_residuals = np.ascontiguousarray(
        residual_pq.decode(residual_codes),
        dtype=np.float32,
    )

    selected_residual_mse = float(
        np.mean(
            (selected_residuals - decoded_residuals) ** 2
        )
    )

    selected_residual_cosine = np.sum(
        selected_residuals * decoded_residuals,
        axis=1,
    ) / (
        np.linalg.norm(selected_residuals, axis=1)
        * np.linalg.norm(decoded_residuals, axis=1)
        + 1e-12
    )

    code_payload_avg_bytes = (
        len(selected_doc_ids) * residual_m / N_DOCS
    )
    metadata_avg_bytes = (
        len(selected_doc_ids) * SIDECAR_ID_BYTES / N_DOCS
    )
    total_avg_bytes = (
        BASE_PQ_BYTES
        + RESIDUAL_PQ_CODEBOOK_AVG_BYTES
        + code_payload_avg_bytes
        + metadata_avg_bytes
    )

    for rank, doc_idx in enumerate(selected_doc_ids, start=1):
        selection_rows.append({
            "residual_pq_m": int(residual_m),
            "selection_rank": int(rank),
            "document_index": int(doc_idx),
            "document_id": doc_ids[doc_idx],
            "residual_code_bytes": int(residual_m),
        })

    for depth in RESIDUAL_PQ_DEPTHS:
        refined = refine_candidates_with_decoded_residuals(
            candidate_ids=m32_rankings_top100,
            candidate_scores=m32_scores,
            selected_ids=selected_doc_ids,
            decoded_residuals=decoded_residuals,
            depth=depth,
        )

        row = heldout_result_row(
            method="residual_pq_sidecar_m32",
            policy="candidate_distortion_sidecar",
            candidate_depth=int(depth),
            rankings=refined["rankings"],
            refinement_latency_ms_per_query=(
                refined["latency_ms_per_query"]
            ),
            refined_candidate_occurrences=(
                refined["corrected_occurrences"]
            ),
            total_bytes_per_vector=float(total_avg_bytes),
            selected_document_count=len(selected_doc_ids),
        )

        row.update({
            "residual_pq_m": int(residual_m),
            "residual_code_bytes_per_selected_doc": int(residual_m),
            "residual_codebook_bytes": int(
                RESIDUAL_PQ_CODEBOOK_BYTES
            ),
            "residual_codebook_avg_bytes_per_vector": float(
                RESIDUAL_PQ_CODEBOOK_AVG_BYTES
            ),
            "residual_code_payload_avg_bytes_per_vector": float(
                code_payload_avg_bytes
            ),
            "sidecar_metadata_avg_bytes_per_vector": float(
                metadata_avg_bytes
            ),
            "selected_residual_mse": selected_residual_mse,
            "selected_residual_mean_cosine": float(
                np.mean(selected_residual_cosine)
            ),
        })

        sweep_rows.append(row)

sweep_results_df = pd.DataFrame(sweep_rows).sort_values(
    ["residual_pq_m", "candidate_depth"]
).reset_index(drop=True)

sweep_selection_df = pd.DataFrame(selection_rows)

sweep_results_df.to_csv(
    RESIDUAL_PQ_DIR / "residual_pq_sweep_heldout_results.csv",
    index=False,
    encoding="utf-8-sig",
)

sweep_selection_df.to_csv(
    RESIDUAL_PQ_DIR / "residual_pq_sweep_selected_documents.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison_columns = [
    "method",
    "policy",
    "residual_pq_m",
    "candidate_depth",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
    "mean_top10_overlap_with_exact",
    "heldout_recovered_drop_events",
    "heldout_relevant_drop_recovery_rate",
    "selected_residual_mse",
    "selected_residual_mean_cosine",
    "total_bytes_per_vector",
    "refinement_latency_ms_per_query",
]

fp16_reference_df = candidate_distortion_results_df[
    candidate_distortion_results_df[
        "candidate_depth"
    ].isin(RESIDUAL_PQ_DEPTHS)
].copy()

fp16_reference_df["residual_pq_m"] = np.nan
fp16_reference_df["selected_residual_mse"] = 0.0
fp16_reference_df["selected_residual_mean_cosine"] = 1.0

oracle_reference_df = oracle_results_df[
    oracle_results_df["candidate_depth"].isin(
        RESIDUAL_PQ_DEPTHS
    )
].copy()

oracle_reference_df["residual_pq_m"] = np.nan
oracle_reference_df["selected_residual_mse"] = 0.0
oracle_reference_df["selected_residual_mean_cosine"] = 1.0

residual_pq_comparison_df = pd.concat(
    [
        sweep_results_df,
        fp16_reference_df,
        oracle_reference_df,
    ],
    ignore_index=True,
    sort=False,
).sort_values(
    ["method", "candidate_depth", "residual_pq_m"],
    na_position="last",
).reset_index(drop=True)

residual_pq_comparison_df.to_csv(
    RESIDUAL_PQ_DIR / "residual_pq_sweep_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# Residual-PQ Diagnostic Code-Size Sweep",
    "",
    "Selected documents are fixed to the Cell 13 candidate-distortion",
    "allocation. This is not a fixed-budget experiment.",
    "",
    "Residual-PQ codebook storage is included in total bytes/vector.",
    "",
    "## Held-out residual-PQ results",
    "",
    sweep_results_df[comparison_columns].to_markdown(index=False),
    "",
    "## FP16 sidecar and oracle references",
    "",
    residual_pq_comparison_df[
        comparison_columns
    ].to_markdown(index=False),
]

(RESIDUAL_PQ_DIR / "residual_pq_sweep_audit.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("\nHeld-out residual-PQ sweep")
display(sweep_results_df[comparison_columns])

print("\nSaved:")
for filename in [
    "residual_pq_sweep_heldout_results.csv",
    "residual_pq_sweep_selected_documents.csv",
    "residual_pq_sweep_comparison.csv",
    "residual_pq_sweep_audit.md",
]:
    print("-", RESIDUAL_PQ_DIR / filename)


In [ ]:
# 16. Fixed-budget residual-PQ allocation experiment
#
# Main experiment:
# Base IVF-PQ M=32 with a strict 48 bytes/vector total storage target.
#
# Storage includes:
# - base PQ code: 32 bytes/vector
# - one amortized residual-PQ codebook
# - residual code payload for selected documents
# - 4-byte document ID metadata for selected documents
#
# Policies:
# - random
# - reconstruction error
# - calibration relevant-drop risk
# - calibration candidate-distortion
#
# Evaluation uses held-out FiQA only.

FIXED_BUDGET_TOTAL_BYTES = 48.0
FIXED_BUDGET_RESIDUAL_M_VALUES = [8, 16, 32]
FIXED_BUDGET_DEPTHS = [20, 50]
FIXED_BUDGET_POLICIES = [
    "random_sidecar",
    "reconstruction_error_sidecar",
    "calibration_drop_risk_sidecar",
    "calibration_candidate_distortion_sidecar",
]
FIXED_BUDGET_SEED = RANDOM_SEED + 16

FIXED_BUDGET_DIR = M32_DIR / "fixed_budget_residual_pq"
FIXED_BUDGET_DIR.mkdir(parents=True, exist_ok=True)

if "candidate_distortion_score" not in globals():
    raise RuntimeError(
        "Cell 13 variables are missing. Run Cell 13 before Cell 16."
    )

if "m32_residuals" not in globals():
    raise RuntimeError(
        "Residual vectors are missing. Run Cell 11 before Cell 16."
    )

if "residual_train_vectors" not in globals():
    raise RuntimeError(
        "Residual-PQ train vectors are missing. Run Cell 15 before Cell 16."
    )

fixed_budget_rng = np.random.default_rng(FIXED_BUDGET_SEED)

def stable_descending_order(
    primary_score,
    secondary_score=None,
    seed_offset=0,
):
    tie_rng = np.random.default_rng(
        FIXED_BUDGET_SEED + seed_offset
    )
    tie_break = tie_rng.permutation(N_DOCS)

    if secondary_score is None:
        return np.lexsort((
            tie_break,
            -np.asarray(primary_score, dtype=np.float64),
        ))

    return np.lexsort((
        tie_break,
        -np.asarray(secondary_score, dtype=np.float64),
        -np.asarray(primary_score, dtype=np.float64),
    ))

candidate_distortion_order = stable_descending_order(
    candidate_distortion_score,
    candidate_frequency,
    seed_offset=101,
)

drop_risk_order = stable_descending_order(
    risk_scores_by_doc,
    seed_offset=102,
)

reconstruction_error_order = stable_descending_order(
    reconstruction_error,
    seed_offset=103,
)

def select_fixed_budget_documents(
    policy,
    selected_count,
    residual_m,
):
    if policy == "random_sidecar":
        rng = np.random.default_rng(
            FIXED_BUDGET_SEED + 1000 + residual_m
        )
        selected = rng.choice(
            N_DOCS,
            size=selected_count,
            replace=False,
        )
        return np.sort(selected.astype(np.int64))

    if policy == "reconstruction_error_sidecar":
        return np.sort(
            reconstruction_error_order[:selected_count].astype(np.int64)
        )

    if policy == "calibration_drop_risk_sidecar":
        return np.sort(
            drop_risk_order[:selected_count].astype(np.int64)
        )

    if policy == "calibration_candidate_distortion_sidecar":
        return np.sort(
            candidate_distortion_order[:selected_count].astype(np.int64)
        )

    raise ValueError(f"Unknown policy: {policy}")

def policy_score_for_doc(policy, doc_idx):
    if policy == "random_sidecar":
        return np.nan

    if policy == "reconstruction_error_sidecar":
        return float(reconstruction_error[doc_idx])

    if policy == "calibration_drop_risk_sidecar":
        return float(risk_scores_by_doc[doc_idx])

    if policy == "calibration_candidate_distortion_sidecar":
        return float(candidate_distortion_score[doc_idx])

    return np.nan

fixed_budget_result_rows = []
fixed_budget_selection_rows = []
fixed_budget_config_rows = []

for residual_m in FIXED_BUDGET_RESIDUAL_M_VALUES:
    residual_codebook_bytes = (
        D * (2 ** RESIDUAL_PQ_NBITS) * 4
    )
    residual_codebook_avg_bytes = (
        residual_codebook_bytes / N_DOCS
    )

    remaining_avg_bytes = (
        FIXED_BUDGET_TOTAL_BYTES
        - BASE_PQ_BYTES
        - residual_codebook_avg_bytes
    )

    entry_bytes = residual_m + SIDECAR_ID_BYTES

    selected_count = int(
        np.floor(
            N_DOCS * remaining_avg_bytes / entry_bytes
        )
    )
    selected_count = max(0, min(N_DOCS, selected_count))

    code_payload_avg_bytes = (
        selected_count * residual_m / N_DOCS
    )
    metadata_avg_bytes = (
        selected_count * SIDECAR_ID_BYTES / N_DOCS
    )
    actual_total_bytes = (
        BASE_PQ_BYTES
        + residual_codebook_avg_bytes
        + code_payload_avg_bytes
        + metadata_avg_bytes
    )

    if actual_total_bytes > FIXED_BUDGET_TOTAL_BYTES + 1e-9:
        raise RuntimeError(
            "Storage budget exceeded unexpectedly: "
            f"{actual_total_bytes:.6f}"
        )

    print(
        "\nResidual-PQ M_r="
        f"{residual_m}: selected={selected_count:,}, "
        f"total={actual_total_bytes:.4f} bytes/vector"
    )

    residual_pq = faiss.ProductQuantizer(
        D,
        residual_m,
        RESIDUAL_PQ_NBITS,
    )

    try:
        residual_pq.cp.niter = 15
        residual_pq.cp.seed = FIXED_BUDGET_SEED + residual_m
    except Exception:
        pass

    residual_pq.train(residual_train_vectors)

    for policy in FIXED_BUDGET_POLICIES:
        selected_ids = select_fixed_budget_documents(
            policy=policy,
            selected_count=selected_count,
            residual_m=residual_m,
        )

        selected_residuals = np.ascontiguousarray(
            m32_residuals[selected_ids],
            dtype=np.float32,
        )

        residual_codes = residual_pq.compute_codes(
            selected_residuals
        )

        decoded_residuals = np.ascontiguousarray(
            residual_pq.decode(residual_codes),
            dtype=np.float32,
        )

        selected_residual_mse = float(
            np.mean(
                (selected_residuals - decoded_residuals) ** 2
            )
        )

        selected_residual_cosine = np.sum(
            selected_residuals * decoded_residuals,
            axis=1,
        ) / (
            np.linalg.norm(selected_residuals, axis=1)
            * np.linalg.norm(decoded_residuals, axis=1)
            + 1e-12
        )

        for rank, doc_idx in enumerate(selected_ids, start=1):
            fixed_budget_selection_rows.append({
                "residual_pq_m": int(residual_m),
                "policy": policy,
                "selection_rank": int(rank),
                "document_index": int(doc_idx),
                "document_id": doc_ids[doc_idx],
                "allocation_score": policy_score_for_doc(
                    policy,
                    doc_idx,
                ),
                "candidate_distortion_score": float(
                    candidate_distortion_score[doc_idx]
                ),
                "drop_risk_score": float(
                    risk_scores_by_doc[doc_idx]
                ),
                "reconstruction_error_l2_sq": float(
                    reconstruction_error[doc_idx]
                ),
            })

        for depth in FIXED_BUDGET_DEPTHS:
            refined = refine_candidates_with_decoded_residuals(
                candidate_ids=m32_rankings_top100,
                candidate_scores=m32_scores,
                selected_ids=selected_ids,
                decoded_residuals=decoded_residuals,
                depth=depth,
            )

            row = heldout_result_row(
                method="fixed_budget_residual_pq_m32",
                policy=policy,
                candidate_depth=int(depth),
                rankings=refined["rankings"],
                refinement_latency_ms_per_query=(
                    refined["latency_ms_per_query"]
                ),
                refined_candidate_occurrences=(
                    refined["corrected_occurrences"]
                ),
                total_bytes_per_vector=float(actual_total_bytes),
                selected_document_count=int(selected_count),
            )

            row.update({
                "residual_pq_m": int(residual_m),
                "residual_codebook_bytes": int(
                    residual_codebook_bytes
                ),
                "residual_codebook_avg_bytes_per_vector": float(
                    residual_codebook_avg_bytes
                ),
                "residual_code_payload_avg_bytes_per_vector": float(
                    code_payload_avg_bytes
                ),
                "sidecar_metadata_avg_bytes_per_vector": float(
                    metadata_avg_bytes
                ),
                "selected_fraction": float(
                    selected_count / N_DOCS
                ),
                "selected_residual_mse": selected_residual_mse,
                "selected_residual_mean_cosine": float(
                    np.mean(selected_residual_cosine)
                ),
                "storage_budget_target_bytes_per_vector": float(
                    FIXED_BUDGET_TOTAL_BYTES
                ),
            })

            fixed_budget_result_rows.append(row)

        fixed_budget_config_rows.append({
            "residual_pq_m": int(residual_m),
            "policy": policy,
            "selected_document_count": int(selected_count),
            "selected_fraction": float(selected_count / N_DOCS),
            "base_pq_bytes_per_vector": float(BASE_PQ_BYTES),
            "residual_codebook_bytes": int(
                residual_codebook_bytes
            ),
            "residual_codebook_avg_bytes_per_vector": float(
                residual_codebook_avg_bytes
            ),
            "residual_code_payload_avg_bytes_per_vector": float(
                code_payload_avg_bytes
            ),
            "sidecar_metadata_avg_bytes_per_vector": float(
                metadata_avg_bytes
            ),
            "total_bytes_per_vector": float(actual_total_bytes),
            "storage_budget_target_bytes_per_vector": float(
                FIXED_BUDGET_TOTAL_BYTES
            ),
        })

fixed_budget_results_df = pd.DataFrame(
    fixed_budget_result_rows
).sort_values(
    [
        "residual_pq_m",
        "policy",
        "candidate_depth",
    ]
).reset_index(drop=True)

fixed_budget_selection_df = pd.DataFrame(
    fixed_budget_selection_rows
)

fixed_budget_config_df = pd.DataFrame(
    fixed_budget_config_rows
).sort_values(
    ["residual_pq_m", "policy"]
).reset_index(drop=True)

fixed_budget_results_df.to_csv(
    FIXED_BUDGET_DIR / "fixed_budget_residual_pq_heldout_results.csv",
    index=False,
    encoding="utf-8-sig",
)

fixed_budget_selection_df.to_csv(
    FIXED_BUDGET_DIR / "fixed_budget_residual_pq_selected_documents.csv",
    index=False,
    encoding="utf-8-sig",
)

fixed_budget_config_df.to_csv(
    FIXED_BUDGET_DIR / "fixed_budget_residual_pq_storage_config.csv",
    index=False,
    encoding="utf-8-sig",
)

reference_df = pd.concat(
    [
        results_df[
            results_df["policy"].isin(
                [
                    "none",
                    "uniform_code_budget",
                    "calibration_candidate_distortion_sidecar",
                ]
            )
        ].copy(),
        fixed_budget_results_df.copy(),
    ],
    ignore_index=True,
    sort=False,
)

reference_df.to_csv(
    FIXED_BUDGET_DIR / "fixed_budget_residual_pq_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

display_columns = [
    "method",
    "policy",
    "residual_pq_m",
    "candidate_depth",
    "selected_document_count",
    "selected_fraction",
    "total_bytes_per_vector",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
    "mean_top10_overlap_with_exact",
    "heldout_recovered_drop_events",
    "heldout_relevant_drop_recovery_rate",
    "refinement_latency_ms_per_query",
]

summary_lines = [
    "# Fixed-Budget Residual-PQ Allocation Experiment",
    "",
    "All Residual-PQ methods use a strict target of 48 bytes/vector.",
    "Storage includes base M=32 PQ, amortized residual-PQ codebook,",
    "residual code payload, and document-ID metadata.",
    "",
    "## Storage configuration",
    "",
    fixed_budget_config_df.to_markdown(index=False),
    "",
    "## Held-out FiQA results",
    "",
    fixed_budget_results_df[
        display_columns
    ].to_markdown(index=False),
    "",
    "## Comparison with base / uniform M=48 / FP16 sparse sidecar",
    "",
    reference_df[
        [
            col for col in display_columns
            if col in reference_df.columns
        ]
    ].to_markdown(index=False),
]

(FIXED_BUDGET_DIR / "fixed_budget_residual_pq_audit.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("\nFixed-budget storage configuration")
display(fixed_budget_config_df)

print("\nHeld-out fixed-budget residual-PQ results")
display(fixed_budget_results_df[display_columns])

print("\nSaved:")
for filename in [
    "fixed_budget_residual_pq_heldout_results.csv",
    "fixed_budget_residual_pq_selected_documents.csv",
    "fixed_budget_residual_pq_storage_config.csv",
    "fixed_budget_residual_pq_comparison.csv",
    "fixed_budget_residual_pq_audit.md",
]:
    print("-", FIXED_BUDGET_DIR / filename)


In [ ]:
# 17. Paired bootstrap significance for fixed-budget residual PQ
#
# Rebuilds only the selected reconstruction-error residual-PQ variants
# needed for paired held-out bootstrap comparisons.
#
# Selection is unchanged from Cell 16. Held-out qrels are used only for
# evaluation, never for sidecar allocation.

BOOTSTRAP_RESAMPLES = 10000
BOOTSTRAP_BATCH_SIZE = 1000
BOOTSTRAP_SEED = RANDOM_SEED + 17
BOOTSTRAP_DEPTH = 50
BOOTSTRAP_RESIDUAL_M_VALUES = [8, 16, 32]

BOOTSTRAP_DIR = FIXED_BUDGET_DIR / "bootstrap_significance"
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)

required_globals = [
    "m32_rankings_top100",
    "m48_rankings_top100",
    "m32_residuals",
    "reconstruction_error_order",
    "residual_train_vectors",
    "fixed_budget_config_df",
    "heldout_mask",
]

missing = [
    name for name in required_globals
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required variables. Run Cells 11, 15, and 16 first: "
        + ", ".join(missing)
    )

def build_reconstruction_error_residual_pq_rankings(residual_m):
    config_row = fixed_budget_config_df[
        fixed_budget_config_df["residual_pq_m"] == residual_m
    ].iloc[0]

    selected_count = int(config_row["selected_document_count"])
    selected_ids = np.sort(
        reconstruction_error_order[:selected_count].astype(np.int64)
    )

    residual_pq = faiss.ProductQuantizer(
        D,
        residual_m,
        RESIDUAL_PQ_NBITS,
    )

    try:
        residual_pq.cp.niter = 15
        residual_pq.cp.seed = FIXED_BUDGET_SEED + residual_m
    except Exception:
        pass

    residual_pq.train(residual_train_vectors)

    selected_residuals = np.ascontiguousarray(
        m32_residuals[selected_ids],
        dtype=np.float32,
    )

    residual_codes = residual_pq.compute_codes(selected_residuals)

    decoded_residuals = np.ascontiguousarray(
        residual_pq.decode(residual_codes),
        dtype=np.float32,
    )

    refined = refine_candidates_with_decoded_residuals(
        candidate_ids=m32_rankings_top100,
        candidate_scores=m32_scores,
        selected_ids=selected_ids,
        decoded_residuals=decoded_residuals,
        depth=BOOTSTRAP_DEPTH,
    )

    return refined["rankings"]


method_rankings = {
    "base_ivfpq_m32": m32_rankings_top100[:, :FINAL_K].astype(
        np.int64,
        copy=True,
    ),
    "uniform_ivfpq_m48": m48_rankings_top100[:, :FINAL_K].astype(
        np.int64,
        copy=True,
    ),
}

for residual_m in BOOTSTRAP_RESIDUAL_M_VALUES:
    print(
        "Rebuilding reconstruction-error residual-PQ "
        f"M_r={residual_m}, Top-{BOOTSTRAP_DEPTH}"
    )

    method_rankings[
        f"residual_pq_{residual_m}b_reconstruction_error_top50"
    ] = build_reconstruction_error_residual_pq_rankings(
        residual_m
    )

metric_names = [
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
]

heldout_positions = np.flatnonzero(heldout_mask)

per_query_rows = []
metric_arrays_by_method = {}

for method_name, rankings in method_rankings.items():
    metrics = per_query_metrics(
        rankings,
        query_ids,
        doc_ids,
        qrels,
        FINAL_K,
    )

    metric_arrays_by_method[method_name] = {
        metric_name: np.asarray(metrics[metric_name])[heldout_mask]
        for metric_name in metric_names
    }

    for local_pos, query_pos in enumerate(heldout_positions):
        row = {
            "method": method_name,
            "query_position": int(query_pos),
            "query_id": query_ids[query_pos],
        }

        for metric_name in metric_names:
            row[metric_name] = float(
                metric_arrays_by_method[method_name][metric_name][local_pos]
            )

        per_query_rows.append(row)

per_query_bootstrap_df = pd.DataFrame(per_query_rows)

comparison_specs = [
    (
        "residual_pq_8b_reconstruction_error_top50",
        "base_ivfpq_m32",
    ),
    (
        "residual_pq_16b_reconstruction_error_top50",
        "base_ivfpq_m32",
    ),
    (
        "residual_pq_32b_reconstruction_error_top50",
        "base_ivfpq_m32",
    ),
    (
        "residual_pq_16b_reconstruction_error_top50",
        "residual_pq_8b_reconstruction_error_top50",
    ),
    (
        "residual_pq_16b_reconstruction_error_top50",
        "residual_pq_32b_reconstruction_error_top50",
    ),
    (
        "uniform_ivfpq_m48",
        "residual_pq_16b_reconstruction_error_top50",
    ),
]

def paired_bootstrap_ci(
    values_a,
    values_b,
    rng,
    n_resamples,
    batch_size,
):
    values_a = np.asarray(values_a, dtype=np.float64)
    values_b = np.asarray(values_b, dtype=np.float64)

    if values_a.shape != values_b.shape:
        raise ValueError("Paired metric arrays must have equal shape.")

    n_queries = len(values_a)
    point_delta = float(np.mean(values_a - values_b))

    bootstrap_deltas = []

    remaining = n_resamples

    while remaining > 0:
        current_batch = min(batch_size, remaining)

        indices = rng.integers(
            0,
            n_queries,
            size=(current_batch, n_queries),
        )

        sampled_delta = (
            values_a[indices].mean(axis=1)
            - values_b[indices].mean(axis=1)
        )

        bootstrap_deltas.append(sampled_delta)
        remaining -= current_batch

    bootstrap_deltas = np.concatenate(bootstrap_deltas)

    ci_low, ci_high = np.percentile(
        bootstrap_deltas,
        [2.5, 97.5],
    )

    return {
        "point_delta": point_delta,
        "ci_low_95": float(ci_low),
        "ci_high_95": float(ci_high),
        "ci_excludes_zero": bool(
            ci_low > 0.0 or ci_high < 0.0
        ),
    }

bootstrap_rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []

for method_a, method_b in comparison_specs:
    for metric_name in metric_names:
        result = paired_bootstrap_ci(
            metric_arrays_by_method[method_a][metric_name],
            metric_arrays_by_method[method_b][metric_name],
            bootstrap_rng,
            BOOTSTRAP_RESAMPLES,
            BOOTSTRAP_BATCH_SIZE,
        )

        bootstrap_rows.append({
            "method_a": method_a,
            "method_b": method_b,
            "metric": metric_name,
            "heldout_query_count": int(len(heldout_positions)),
            "bootstrap_resamples": int(BOOTSTRAP_RESAMPLES),
            **result,
        })

bootstrap_results_df = pd.DataFrame(bootstrap_rows)

bootstrap_results_df.to_csv(
    BOOTSTRAP_DIR / "fixed_budget_residual_pq_bootstrap.csv",
    index=False,
    encoding="utf-8-sig",
)

per_query_bootstrap_df.to_csv(
    BOOTSTRAP_DIR / "fixed_budget_residual_pq_per_query_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# Paired Bootstrap Significance: Fixed-Budget Residual-PQ",
    "",
    "All comparisons use the held-out FiQA query set.",
    "Delta is defined as method_a minus method_b.",
    "",
    bootstrap_results_df.to_markdown(index=False),
    "",
    "Interpretation:",
    "- A CI excluding zero indicates a directional difference under this paired bootstrap procedure.",
    "- The test validates the current held-out split; it is not cross-dataset evidence.",
]

(BOOTSTRAP_DIR / "fixed_budget_residual_pq_bootstrap.md").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("Paired bootstrap results")
display(bootstrap_results_df)

print("\nSaved:")
for filename in [
    "fixed_budget_residual_pq_bootstrap.csv",
    "fixed_budget_residual_pq_per_query_metrics.csv",
    "fixed_budget_residual_pq_bootstrap.md",
]:
    print("-", BOOTSTRAP_DIR / filename)


In [ ]:
# 18. Compact Residual-PQ Sidecar Experiment
#
# This experiment keeps the M=32 IVF-PQ base index and the strict
# 48 bytes/vector budget, but replaces per-document uint32 IDs with:
#
# - a 1-bit selection bitmap indexed by Faiss / corpus internal row ID
# - uint32 rank-prefix values per 256-document block
# - residual codes stored in ascending internal-ID order
#
# It also stores the trained residual-PQ centroid table as FP16.
#
# Evaluated configurations:
# 1. Compact-8bit: M_r=16, nbits=8, FP16 codebook
# 2. Compact-4bit: M_r=32, nbits=4, FP16 codebook
#
# Allocation: reconstruction error
# Candidate refinement: Top-50
# Evaluation: held-out FiQA only

from math import ceil

COMPACT_SIDECAR_DIR = (
    M32_DIR / "compact_residual_pq_sidecar"
)
COMPACT_SIDECAR_DIR.mkdir(parents=True, exist_ok=True)

COMPACT_TOTAL_BYTES_PER_VECTOR = 48.0
COMPACT_BASE_BYTES_PER_VECTOR = 32.0
COMPACT_BLOCK_SIZE = 256
COMPACT_ALIGNMENT_BYTES = 64
COMPACT_HEADER_BYTES = 32
COMPACT_DEPTH = 50
COMPACT_POLICY = "reconstruction_error_sidecar"
COMPACT_SEED = FIXED_BUDGET_SEED + 18

COMPACT_CONFIGS = [
    {
        "layout_name": "compact_8bit_m16_fp16_codebook",
        "residual_pq_m": 16,
        "residual_pq_nbits": 8,
    },
    {
        "layout_name": "compact_4bit_m32_fp16_codebook",
        "residual_pq_m": 32,
        "residual_pq_nbits": 4,
    },
]

required_globals = [
    "m32_rankings_top100",
    "m32_scores",
    "m32_residuals",
    "reconstruction_error_order",
    "residual_train_vectors",
    "queries_cpu",
    "heldout_mask",
]

missing = [
    name for name in required_globals
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 11, 15, and 16 before Cell 18. Missing: "
        + ", ".join(missing)
    )


def align_up(byte_count, alignment=COMPACT_ALIGNMENT_BYTES):
    return int(
        ceil(byte_count / alignment) * alignment
    )


def compact_layout_bytes(n_docs, block_size=COMPACT_BLOCK_SIZE):
    bitmap_bytes = int(ceil(n_docs / 8))
    block_count = int(ceil(n_docs / block_size))

    # Prefix length = block_count + terminal total.
    prefix_bytes = int((block_count + 1) * 4)

    raw_index_bytes = (
        COMPACT_HEADER_BYTES
        + bitmap_bytes
        + prefix_bytes
    )

    return {
        "bitmap_bytes": bitmap_bytes,
        "rank_prefix_bytes": prefix_bytes,
        "raw_selection_index_bytes": raw_index_bytes,
        "serialized_selection_index_bytes": align_up(
            raw_index_bytes
        ),
        "block_count": block_count,
    }


def build_bitmap_and_prefix(selected_ids, n_docs, block_size):
    selected_ids = np.asarray(
        selected_ids,
        dtype=np.int64,
    )

    if len(selected_ids) > 1 and np.any(
        selected_ids[1:] <= selected_ids[:-1]
    ):
        raise ValueError(
            "selected_ids must be sorted and unique."
        )

    bitmap = np.zeros(
        int(ceil(n_docs / 8)),
        dtype=np.uint8,
    )

    for doc_id in selected_ids:
        bitmap[int(doc_id) // 8] |= np.uint8(
            1 << (int(doc_id) % 8)
        )

    block_count = int(ceil(n_docs / block_size))
    prefix = np.zeros(
        block_count + 1,
        dtype=np.uint32,
    )

    running = 0

    for block in range(block_count):
        prefix[block] = running

        start = block * block_size
        end = min(start + block_size, n_docs)

        for doc_id in range(start, end):
            byte_index = doc_id // 8
            bit_index = doc_id % 8

            if (int(bitmap[byte_index]) >> bit_index) & 1:
                running += 1

    prefix[-1] = running

    if int(prefix[-1]) != len(selected_ids):
        raise AssertionError(
            "Compact prefix total does not match selected IDs."
        )

    return bitmap, prefix


def compact_lookup_slot(
    bitmap,
    prefix,
    doc_id,
    block_size=COMPACT_BLOCK_SIZE,
):
    byte_index = doc_id // 8
    bit_index = doc_id % 8

    if not ((int(bitmap[byte_index]) >> bit_index) & 1):
        return None

    block = doc_id // block_size
    block_start = block * block_size
    slot = int(prefix[block])

    for previous_id in range(block_start, doc_id):
        previous_byte = previous_id // 8
        previous_bit = previous_id % 8

        if (
            int(bitmap[previous_byte])
            >> previous_bit
        ) & 1:
            slot += 1

    return slot


def selected_count_for_budget(
    n_docs,
    codebook_serialized_bytes,
    selection_index_serialized_bytes,
    residual_payload_bytes,
):
    total_budget_bytes = int(
        round(
            COMPACT_TOTAL_BYTES_PER_VECTOR
            * n_docs
        )
    )

    base_bytes = int(
        round(
            COMPACT_BASE_BYTES_PER_VECTOR
            * n_docs
        )
    )

    available_payload_bytes = (
        total_budget_bytes
        - base_bytes
        - codebook_serialized_bytes
        - selection_index_serialized_bytes
    )

    return max(
        0,
        min(
            n_docs,
            available_payload_bytes
            // residual_payload_bytes,
        ),
    )


def refine_with_compact_sidecar(
    candidate_ids,
    candidate_scores,
    bitmap,
    prefix,
    runtime_slot_by_doc_id,
    decoded_residuals,
    depth,
):
    refined_rankings = np.full(
        (candidate_ids.shape[0], FINAL_K),
        -1,
        dtype=np.int64,
    )

    corrected_occurrences = 0
    start = time.perf_counter()

    for query_pos in range(candidate_ids.shape[0]):
        ids = candidate_ids[
            query_pos,
            :depth,
        ].astype(np.int64, copy=True)

        scores = candidate_scores[
            query_pos,
            :depth,
        ].astype(np.float32, copy=True)

        query_vector = queries_cpu[query_pos]

        for candidate_pos, doc_id in enumerate(ids):
            doc_id = int(doc_id)

            if doc_id < 0:
                continue

            # Evaluation-only accelerator. The deployable layout remains
            # bitmap + rank-prefix; this dense array is not storage-accounted.
            slot = int(runtime_slot_by_doc_id[doc_id])

            if slot < 0:
                continue

            scores[candidate_pos] += float(
                np.dot(
                    query_vector,
                    decoded_residuals[slot],
                )
            )

            corrected_occurrences += 1

        order = np.argsort(
            -scores,
            kind="stable",
        )

        refined_rankings[query_pos] = ids[
            order[:FINAL_K]
        ]

    elapsed_seconds = time.perf_counter() - start

    return {
        "rankings": refined_rankings,
        "corrected_occurrences": int(
            corrected_occurrences
        ),
        "latency_ms_per_query": (
            elapsed_seconds
            * 1000.0
            / candidate_ids.shape[0]
        ),
    }


compact_result_rows = []
compact_storage_rows = []
compact_selection_rows = []
compact_rankings_by_layout = {}

layout = compact_layout_bytes(N_DOCS)

for config in COMPACT_CONFIGS:
    layout_name = config["layout_name"]
    residual_m = int(config["residual_pq_m"])
    residual_nbits = int(
        config["residual_pq_nbits"]
    )

    codebook_raw_bytes = (
        D
        * (2 ** residual_nbits)
        * 2
    )

    codebook_serialized_bytes = align_up(
        COMPACT_HEADER_BYTES
        + codebook_raw_bytes
    )

    payload_bytes_per_selected_doc = int(
        ceil(residual_m * residual_nbits / 8)
    )

    selected_count = selected_count_for_budget(
        n_docs=N_DOCS,
        codebook_serialized_bytes=(
            codebook_serialized_bytes
        ),
        selection_index_serialized_bytes=(
            layout[
                "serialized_selection_index_bytes"
            ]
        ),
        residual_payload_bytes=(
            payload_bytes_per_selected_doc
        ),
    )

    selected_ids = np.sort(
        reconstruction_error_order[
            :selected_count
        ].astype(np.int64)
    )

    print(
        f"\n{layout_name}: "
        f"selected={selected_count:,} "
        f"({selected_count / N_DOCS:.2%})"
    )

    residual_pq = faiss.ProductQuantizer(
        D,
        residual_m,
        residual_nbits,
    )

    try:
        residual_pq.cp.niter = 15
        residual_pq.cp.seed = (
            COMPACT_SEED
            + residual_m
            + residual_nbits
        )
    except Exception:
        pass

    residual_pq.train(residual_train_vectors)

    selected_residuals = np.ascontiguousarray(
        m32_residuals[selected_ids],
        dtype=np.float32,
    )

    # Codes are assigned using the trained FP32 codebook.
    residual_codes = residual_pq.compute_codes(
        selected_residuals
    )

    # Serving codebook contract:
    # serialize centroids as FP16, restore as FP32 only for decoding.
    trained_centroids = faiss.vector_to_array(
        residual_pq.centroids
    ).astype(np.float32, copy=True)

    fp16_centroids = np.ascontiguousarray(
        trained_centroids.astype(np.float16)
    )

    serving_centroids = np.ascontiguousarray(
        fp16_centroids.astype(np.float32)
    )

    faiss.copy_array_to_vector(
        serving_centroids,
        residual_pq.centroids,
    )

    decoded_residuals = np.ascontiguousarray(
        residual_pq.decode(residual_codes),
        dtype=np.float32,
    )

    bitmap, prefix = build_bitmap_and_prefix(
        selected_ids=selected_ids,
        n_docs=N_DOCS,
        block_size=COMPACT_BLOCK_SIZE,
    )

    # Evaluation-only runtime accelerator. It is intentionally excluded
    # from strict deployable storage accounting.
    runtime_slot_by_doc_id = np.full(
        N_DOCS,
        -1,
        dtype=np.int32,
    )
    runtime_slot_by_doc_id[selected_ids] = np.arange(
        selected_count,
        dtype=np.int32,
    )

    # Validate compact slots against the code order.
    validation_ids = np.concatenate(
        [
            selected_ids[: min(100, len(selected_ids))],
            np.random.default_rng(
                COMPACT_SEED + residual_m
            ).integers(
                0,
                N_DOCS,
                size=1000,
                dtype=np.int64,
            ),
        ]
    )

    legacy_lookup = {
        int(doc_id): position
        for position, doc_id in enumerate(selected_ids)
    }

    for doc_id in validation_ids:
        compact_slot = compact_lookup_slot(
            bitmap=bitmap,
            prefix=prefix,
            doc_id=int(doc_id),
        )

        legacy_slot = legacy_lookup.get(
            int(doc_id)
        )

        runtime_slot = int(
            runtime_slot_by_doc_id[int(doc_id)]
        )
        runtime_slot = None if runtime_slot < 0 else runtime_slot

        if compact_slot != legacy_slot or runtime_slot != legacy_slot:
            raise AssertionError(
                "Compact lookup mismatch for "
                f"doc_id={doc_id}: "
                f"compact={compact_slot}, "
                f"runtime={runtime_slot}, "
                f"legacy={legacy_slot}"
            )

    selected_residual_mse = float(
        np.mean(
            (
                selected_residuals
                - decoded_residuals
            ) ** 2
        )
    )

    selected_residual_cosine = np.sum(
        selected_residuals
        * decoded_residuals,
        axis=1,
    ) / (
        np.linalg.norm(
            selected_residuals,
            axis=1,
        )
        * np.linalg.norm(
            decoded_residuals,
            axis=1,
        )
        + 1e-12
    )

    actual_total_bytes = (
        COMPACT_BASE_BYTES_PER_VECTOR
        * N_DOCS
        + codebook_serialized_bytes
        + layout[
            "serialized_selection_index_bytes"
        ]
        + selected_count
        * payload_bytes_per_selected_doc
    ) / N_DOCS

    if (
        actual_total_bytes
        > COMPACT_TOTAL_BYTES_PER_VECTOR + 1e-9
    ):
        raise RuntimeError(
            "Compact storage budget exceeded: "
            f"{actual_total_bytes:.8f}"
        )

    refined = refine_with_compact_sidecar(
        candidate_ids=m32_rankings_top100,
        candidate_scores=m32_scores,
        bitmap=bitmap,
        prefix=prefix,
        runtime_slot_by_doc_id=runtime_slot_by_doc_id,
        decoded_residuals=decoded_residuals,
        depth=COMPACT_DEPTH,
    )

    compact_rankings_by_layout[layout_name] = (
        refined["rankings"].copy()
    )

    result_row = heldout_result_row(
        method="compact_residual_pq_m32",
        policy=COMPACT_POLICY,
        candidate_depth=COMPACT_DEPTH,
        rankings=refined["rankings"],
        refinement_latency_ms_per_query=(
            refined["latency_ms_per_query"]
        ),
        refined_candidate_occurrences=(
            refined["corrected_occurrences"]
        ),
        total_bytes_per_vector=float(
            actual_total_bytes
        ),
        selected_document_count=int(
            selected_count
        ),
    )

    result_row.update({
        "layout_name": layout_name,
        "residual_pq_m": residual_m,
        "residual_pq_nbits": residual_nbits,
        "residual_code_payload_bytes_per_selected_doc": int(
            payload_bytes_per_selected_doc
        ),
        "residual_codebook_dtype": "float16",
        "residual_codebook_raw_bytes": int(
            codebook_raw_bytes
        ),
        "residual_codebook_serialized_bytes": int(
            codebook_serialized_bytes
        ),
        "selection_bitmap_bytes": int(
            layout["bitmap_bytes"]
        ),
        "selection_rank_prefix_bytes": int(
            layout["rank_prefix_bytes"]
        ),
        "selection_index_serialized_bytes": int(
            layout[
                "serialized_selection_index_bytes"
            ]
        ),
        "residual_code_payload_bytes_per_selected_doc": int(
            payload_bytes_per_selected_doc
        ),
        "residual_code_payload_bytes": int(
            selected_count
            * payload_bytes_per_selected_doc
        ),
        "selected_fraction": float(
            selected_count / N_DOCS
        ),
        "selected_residual_mse": selected_residual_mse,
        "selected_residual_mean_cosine": float(
            np.mean(selected_residual_cosine)
        ),
        "storage_budget_target_bytes_per_vector": float(
            COMPACT_TOTAL_BYTES_PER_VECTOR
        ),
        "lookup_equivalence": "pass",
        "runtime_lookup_mode": (
            "evaluation_dense_slot_map_not_storage_accounted"
        ),
    })

    compact_result_rows.append(result_row)

    compact_storage_rows.append({
        "layout_name": layout_name,
        "residual_pq_m": residual_m,
        "residual_pq_nbits": residual_nbits,
        "selected_document_count": int(
            selected_count
        ),
        "selected_fraction": float(
            selected_count / N_DOCS
        ),
        "base_pq_total_bytes": int(
            COMPACT_BASE_BYTES_PER_VECTOR
            * N_DOCS
        ),
        "residual_codebook_raw_bytes": int(
            codebook_raw_bytes
        ),
        "residual_codebook_serialized_bytes": int(
            codebook_serialized_bytes
        ),
        "selection_bitmap_bytes": int(
            layout["bitmap_bytes"]
        ),
        "selection_rank_prefix_bytes": int(
            layout["rank_prefix_bytes"]
        ),
        "selection_index_serialized_bytes": int(
            layout[
                "serialized_selection_index_bytes"
            ]
        ),
        "residual_code_payload_bytes_per_selected_doc": int(
            payload_bytes_per_selected_doc
        ),
        "residual_code_payload_bytes": int(
            selected_count * payload_bytes_per_selected_doc
        ),
        "total_bytes_per_vector": float(
            actual_total_bytes
        ),
        "storage_budget_target_bytes_per_vector": float(
            COMPACT_TOTAL_BYTES_PER_VECTOR
        ),
    })

    for rank, doc_id in enumerate(
        selected_ids,
        start=1,
    ):
        compact_selection_rows.append({
            "layout_name": layout_name,
            "residual_pq_m": residual_m,
            "residual_pq_nbits": residual_nbits,
            "selection_rank": int(rank),
            "document_index": int(doc_id),
            "document_id": doc_ids[int(doc_id)],
            "reconstruction_error_l2_sq": float(
                reconstruction_error[int(doc_id)]
            ),
        })

compact_results_df = pd.DataFrame(
    compact_result_rows
).sort_values("layout_name").reset_index(drop=True)

compact_storage_df = pd.DataFrame(
    compact_storage_rows
).sort_values("layout_name").reset_index(drop=True)

compact_selection_df = pd.DataFrame(
    compact_selection_rows
).sort_values(
    ["layout_name", "selection_rank"]
).reset_index(drop=True)

compact_results_df.to_csv(
    COMPACT_SIDECAR_DIR
    / "compact_residual_pq_heldout_results.csv",
    index=False,
    encoding="utf-8-sig",
)

compact_storage_df.to_csv(
    COMPACT_SIDECAR_DIR
    / "compact_residual_pq_storage_config.csv",
    index=False,
    encoding="utf-8-sig",
)

compact_selection_df.to_csv(
    COMPACT_SIDECAR_DIR
    / "compact_residual_pq_selected_documents.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison_df = pd.concat(
    [
        fixed_budget_results_df[
            (
                fixed_budget_results_df["policy"]
                == "reconstruction_error_sidecar"
            )
            & (
                fixed_budget_results_df[
                    "candidate_depth"
                ] == COMPACT_DEPTH
            )
        ].copy(),
        compact_results_df.copy(),
    ],
    ignore_index=True,
    sort=False,
)

comparison_df.to_csv(
    COMPACT_SIDECAR_DIR
    / "compact_residual_pq_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# Compact Residual-PQ Sidecar Experiment",
    "",
    "The compact layout replaces per-selected-document IDs with "
    "a bitmap and block-level rank-prefix index keyed by Faiss "
    "internal document IDs.",
    "",
    "The stored residual-PQ centroid table is FP16. "
    "At serving-time it is restored to FP32 only for decoding.",
    "",
    "The recorded refinement latency uses an evaluation-only dense "
    "document-index-to-slot accelerator. It is excluded from strict "
    "deployable storage accounting; bitmap/rank-prefix equivalence is "
    "validated separately.",
    "",
    "## Strict storage accounting",
    "",
    compact_storage_df.to_markdown(index=False),
    "",
    "## Held-out FiQA results",
    "",
    compact_results_df[
        [
            "layout_name",
            "residual_pq_m",
            "residual_pq_nbits",
            "selected_document_count",
            "selected_fraction",
            "total_bytes_per_vector",
            "recall_at_10",
            "mrr_at_10",
            "ndcg_at_10",
            "heldout_recovered_drop_events",
            "heldout_relevant_drop_recovery_rate",
            "selected_residual_mse",
            "selected_residual_mean_cosine",
            "lookup_equivalence",
        ]
    ].to_markdown(index=False),
]

(
    COMPACT_SIDECAR_DIR
    / "compact_residual_pq_audit.md"
).write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("\nCompact storage configuration")
display(compact_storage_df)

print("\nCompact held-out results")
display(
    compact_results_df[
        [
            "layout_name",
            "selected_document_count",
            "selected_fraction",
            "total_bytes_per_vector",
            "recall_at_10",
            "mrr_at_10",
            "ndcg_at_10",
            "heldout_recovered_drop_events",
            "heldout_relevant_drop_recovery_rate",
            "selected_residual_mse",
            "selected_residual_mean_cosine",
            "lookup_equivalence",
        ]
    ]
)

print("\nSaved:")
for filename in [
    "compact_residual_pq_heldout_results.csv",
    "compact_residual_pq_storage_config.csv",
    "compact_residual_pq_selected_documents.csv",
    "compact_residual_pq_comparison.csv",
    "compact_residual_pq_audit.md",
]:
    print("-", COMPACT_SIDECAR_DIR / filename)

In [ ]:
# 19. Paired Bootstrap: Compact Residual-PQ Sidecars
#
# Tests compact sidecars against:
# - base M=32 IVF-PQ
# - legacy fixed-budget 16B Residual-PQ
# - uniform M=48 IVF-PQ
# - each other
#
# Delta is method_a minus method_b.
# Held-out FiQA labels are used only for evaluation.

COMPACT_BOOTSTRAP_RESAMPLES = 10000
COMPACT_BOOTSTRAP_BATCH_SIZE = 500
COMPACT_BOOTSTRAP_SEED = RANDOM_SEED + 19
COMPACT_BOOTSTRAP_DEPTH = 50

COMPACT_BOOTSTRAP_DIR = (
    COMPACT_SIDECAR_DIR / "bootstrap_significance"
)
COMPACT_BOOTSTRAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_globals = [
    "compact_rankings_by_layout",
    "compact_results_df",
    "m32_rankings_top100",
    "m48_rankings_top100",
    "m32_residuals",
    "m32_scores",
    "reconstruction_error_order",
    "residual_train_vectors",
    "fixed_budget_config_df",
    "heldout_mask",
]

missing = [
    name for name in required_globals
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 11, 15, 16, and 18 first. Missing: "
        + ", ".join(missing)
    )


def build_legacy_residual_pq_16b_rankings():
    config_row = fixed_budget_config_df[
        (
            fixed_budget_config_df["residual_pq_m"] == 16
        )
        & (
            fixed_budget_config_df["policy"]
            == "reconstruction_error_sidecar"
        )
    ].iloc[0]

    selected_count = int(
        config_row["selected_document_count"]
    )

    selected_ids = np.sort(
        reconstruction_error_order[
            :selected_count
        ].astype(np.int64)
    )

    residual_pq = faiss.ProductQuantizer(
        D,
        16,
        RESIDUAL_PQ_NBITS,
    )

    try:
        residual_pq.cp.niter = 15
        residual_pq.cp.seed = FIXED_BUDGET_SEED + 16
    except Exception:
        pass

    residual_pq.train(residual_train_vectors)

    selected_residuals = np.ascontiguousarray(
        m32_residuals[selected_ids],
        dtype=np.float32,
    )

    residual_codes = residual_pq.compute_codes(
        selected_residuals
    )

    decoded_residuals = np.ascontiguousarray(
        residual_pq.decode(residual_codes),
        dtype=np.float32,
    )

    refined = refine_candidates_with_decoded_residuals(
        candidate_ids=m32_rankings_top100,
        candidate_scores=m32_scores,
        selected_ids=selected_ids,
        decoded_residuals=decoded_residuals,
        depth=COMPACT_BOOTSTRAP_DEPTH,
    )

    return refined["rankings"]


method_rankings = {
    "base_ivfpq_m32": m32_rankings_top100[
        :, :FINAL_K
    ].astype(np.int64, copy=True),
    "uniform_ivfpq_m48": m48_rankings_top100[
        :, :FINAL_K
    ].astype(np.int64, copy=True),
    "legacy_residual_pq_16b_top50": (
        build_legacy_residual_pq_16b_rankings()
    ),
    "compact_4bit_m32_top50": (
        compact_rankings_by_layout[
            "compact_4bit_m32_fp16_codebook"
        ]
    ),
    "compact_8bit_m16_top50": (
        compact_rankings_by_layout[
            "compact_8bit_m16_fp16_codebook"
        ]
    ),
}

metric_names = [
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
]

heldout_positions = np.flatnonzero(heldout_mask)
metric_arrays_by_method = {}
per_query_rows = []

for method_name, rankings in method_rankings.items():
    metrics = per_query_metrics(
        rankings,
        query_ids,
        doc_ids,
        qrels,
        FINAL_K,
    )

    metric_arrays_by_method[method_name] = {
        metric_name: np.asarray(
            metrics[metric_name],
            dtype=np.float64,
        )[heldout_mask]
        for metric_name in metric_names
    }

    for local_pos, query_pos in enumerate(
        heldout_positions
    ):
        row = {
            "method": method_name,
            "query_position": int(query_pos),
            "query_id": query_ids[query_pos],
        }

        for metric_name in metric_names:
            row[metric_name] = float(
                metric_arrays_by_method[
                    method_name
                ][metric_name][local_pos]
            )

        per_query_rows.append(row)

per_query_df = pd.DataFrame(per_query_rows)

# Verify that retained compact rankings reproduce Cell 18 aggregates.
expected_compact = compact_results_df.set_index(
    "layout_name"
)

validation_map = {
    "compact_4bit_m32_top50": (
        "compact_4bit_m32_fp16_codebook"
    ),
    "compact_8bit_m16_top50": (
        "compact_8bit_m16_fp16_codebook"
    ),
}

for method_name, layout_name in validation_map.items():
    for metric_name in metric_names:
        observed = float(
            metric_arrays_by_method[
                method_name
            ][metric_name].mean()
        )
        expected = float(
            expected_compact.loc[
                layout_name,
                metric_name,
            ]
        )

        if not np.isclose(
            observed,
            expected,
            rtol=0.0,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Compact ranking mismatch for {layout_name}, "
                f"{metric_name}: {observed} != {expected}"
            )

print("Compact aggregate validation: PASS")


def paired_bootstrap(
    values_a,
    values_b,
    rng,
    n_resamples,
    batch_size,
):
    values_a = np.asarray(values_a, dtype=np.float64)
    values_b = np.asarray(values_b, dtype=np.float64)

    if values_a.shape != values_b.shape:
        raise ValueError(
            "Paired metric arrays must have equal shape."
        )

    point_delta = float(np.mean(values_a - values_b))
    n_queries = len(values_a)
    samples = []
    remaining = n_resamples

    while remaining > 0:
        current_batch = min(batch_size, remaining)

        indices = rng.integers(
            0,
            n_queries,
            size=(current_batch, n_queries),
        )

        deltas = (
            values_a[indices].mean(axis=1)
            - values_b[indices].mean(axis=1)
        )

        samples.append(deltas)
        remaining -= current_batch

    samples = np.concatenate(samples)

    ci_low, ci_high = np.percentile(
        samples,
        [2.5, 97.5],
    )

    return {
        "point_delta": point_delta,
        "ci_low_95": float(ci_low),
        "ci_high_95": float(ci_high),
        "ci_excludes_zero": bool(
            ci_low > 0.0 or ci_high < 0.0
        ),
    }


comparison_specs = [
    (
        "compact_8bit_m16_top50",
        "base_ivfpq_m32",
    ),
    (
        "compact_8bit_m16_top50",
        "legacy_residual_pq_16b_top50",
    ),
    (
        "uniform_ivfpq_m48",
        "compact_8bit_m16_top50",
    ),
    (
        "compact_4bit_m32_top50",
        "base_ivfpq_m32",
    ),
    (
        "compact_4bit_m32_top50",
        "compact_8bit_m16_top50",
    ),
    (
        "uniform_ivfpq_m48",
        "compact_4bit_m32_top50",
    ),
]

bootstrap_rows = []

for comparison_index, (method_a, method_b) in enumerate(
    comparison_specs
):
    for metric_index, metric_name in enumerate(metric_names):
        rng = np.random.default_rng(
            COMPACT_BOOTSTRAP_SEED
            + comparison_index * 10
            + metric_index
        )

        result = paired_bootstrap(
            metric_arrays_by_method[method_a][metric_name],
            metric_arrays_by_method[method_b][metric_name],
            rng,
            COMPACT_BOOTSTRAP_RESAMPLES,
            COMPACT_BOOTSTRAP_BATCH_SIZE,
        )

        bootstrap_rows.append({
            "method_a": method_a,
            "method_b": method_b,
            "metric": metric_name,
            "heldout_query_count": int(
                len(heldout_positions)
            ),
            "bootstrap_resamples": int(
                COMPACT_BOOTSTRAP_RESAMPLES
            ),
            **result,
        })

compact_bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

compact_bootstrap_df.to_csv(
    COMPACT_BOOTSTRAP_DIR
    / "compact_residual_pq_bootstrap.csv",
    index=False,
    encoding="utf-8-sig",
)

per_query_df.to_csv(
    COMPACT_BOOTSTRAP_DIR
    / "compact_residual_pq_per_query_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_lines = [
    "# Paired Bootstrap: Compact Residual-PQ Sidecars",
    "",
    "All tests use the held-out FiQA split.",
    "Delta is defined as method_a minus method_b.",
    "",
    compact_bootstrap_df.to_markdown(index=False),
    "",
    "Interpretation:",
    "- A confidence interval excluding zero supports a directional difference on this held-out split.",
    "- This is not cross-dataset evidence.",
]

(
    COMPACT_BOOTSTRAP_DIR
    / "compact_residual_pq_bootstrap.md"
).write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("\nCompact paired-bootstrap results")
display(compact_bootstrap_df)

print("\nSaved:")
for filename in [
    "compact_residual_pq_bootstrap.csv",
    "compact_residual_pq_per_query_metrics.csv",
    "compact_residual_pq_bootstrap.md",
]:
    print("-", COMPACT_BOOTSTRAP_DIR / filename)